# Indoor Localization Using IMU and BLE Beacon Fusion

### A Pedestrian Dead Reckoning and Particle Filter Approach with Building Map Constraints

---

**Group E**

1. Arpitha Thimmegowda
2. Darshan Morkar
3. Venkata Kishore Dannana

**Course:** *Embedded Intelligence - MAI*

**Date:** *22 July 2026*


## 2. Introduction

### 2.1 Indoor Localization Problem
GPS does not work reliably inside buildings because walls and roofs weaken or block the satellite signals. Therefore, our indoor localization system uses the sensors already available on a smartphone together with Bluetooth Low Energy (BLE) beacons to estimate the user's position while walking.

### 2.2 Why We Combine IMU and BLE
Pedestrian Dead Reckoning (PDR) gives a smooth movement estimate over short distances, but small errors in step length and heading accumulate after every step. BLE does not have the same cumulative drift, but its RSSI values fluctuate and cannot provide an exact distance. We combine both because they compensate for each other's weaknesses: PDR tracks the movement between steps, while BLE provides an approximate absolute correction.

### 2.3 Objectives
1. Collect synchronized IMU and BLE data along predefined indoor routes.
2. Detect footsteps from the accelerometer and generate a PDR trajectory.
3. Process the RSSI values from two BLE beacons and use them for position correction.
4. Fuse PDR and BLE in a particle filter while restricting particles to walkable areas.
5. Compare the estimated trajectory with manually recorded reference points.

### 2.4 Pipeline
Raw sensor data → cleaning and synchronization → step detection and PDR → BLE processing and weighting → particle filter with BLE and map constraints → evaluation against reference points.


## 3. Experimental Setup and Data Collection

Four smartphone IMU/BLE recordings from 18-19 July 2026 are configured in the loading cell. R3 includes a staircase transition. Inputs and building assets must be supplied locally; see [data setup](docs/DATA.md). Device field notes conflict (Redmi Note 11 Pro versus Samsung S24 FE for R1/R2/R4); R3 is documented as iQOO Neo 6. Confirm the original field log before asserting device details.


## 4. Experimental Protocol and Measurement Runs

We followed the same basic procedure for R1 to R4:

1. The participant stood at the selected starting point and started the recording. A short settling time was allowed before walking.
2. The participant followed the predefined route at a natural pace and passed the selected doors and checkpoints in order.
3. The time at each checkpoint was manually noted and later used as ground truth, as described in Section 5.
4. At the final checkpoint, the participant paused briefly before stopping the recording.

The runs were not completely identical. They differed in direction, walking speed, recording day, and whether a floor transition was included. R1, R2, and R4 were recorded on 18 July 2026, while R3 was recorded on 19 July 2026.

*(Exact route details and unusual events can be added from the field notes if needed.)*


## 5. Reference Door Timestamps

Manual door/checkpoint references supply run ID, reference ID, time, map position and floor. Start supplies the initial position and floor. Later references are used only for evaluation. Matching subtracts the first cleaned IMU timestamp from PF time, then selects the nearest estimate on the reference floor without a maximum time gap. Missing-floor references are skipped. See [evaluation details](docs/EXPERIMENTS.md).


## 6. Sensors and Data Sources

The pipeline accepts accelerometer, gyroscope, processed orientation, and BLE events. Acceleration is treated as m/s^2, gyro as rad/s, and orientation as degrees. Heading fusion projects gyro onto estimated gravity and integrates using real timestamps. Some signal preparation and BLE windows use future samples, so this notebook is an offline experiment. See [architecture](docs/ARCHITECTURE.md) for timing and coordinate conventions.


## 7. Data Loading

The code below creates the shared in-memory data structures used throughout the notebook. For each run, it reads the raw CSV export, cleans the relevant rows, and separates them into BLE and IMU tables.


In [ ]:
from __future__ import annotations
import pandas as pd
import numpy as np
from pathlib import Path

# In-Memory Storage for Pipeline
GLOBAL_BLE_DATA = {}
GLOBAL_IMU_DATA = {}
GLOBAL_STEPS_DATA = {}
GLOBAL_PDR_DATA = {}
GLOBAL_PF_POSITIONS = {}
GLOBAL_PF_CLOUD = {}
GLOBAL_PF_ERRORS = {}
GLOBAL_PF_ERROR_SUMMARY = pd.DataFrame()


In [ ]:
from __future__ import annotations
"""Generate per-run cleaned BLE and IMU CSV files from raw pickup exports."""

import csv
import re
from pathlib import Path


ROOT = Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

RUN_FILES = {
    "R1":"pickup_export_20260718_113424.csv",
    "R2": "pickup_export_20260718_115144.csv",
    "R3": "pickup_export_20260719_151451.csv",
    "R4": "pickup_export_20260718_121428.csv",
}

BLE_SOURCES = {"beacon", "ble_rssi", "live_ble_snapshot"}
IMU_SENSORS = {"accel", "gyro", "mag", "imu_processed"}
ARRIVE_RE = re.compile(r"arrive_emi\d+")
STARTUP_EMPTY_ID_TRIM_SEC = 0.0


def parse_float(value: str) -> float | None:
    """Parse a string as a float, returning None if it is missing or invalid."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def extract_arrive_beacon_id(row: dict[str, str]) -> str | None:
    """Extract the arrive_emi<N> beacon identifier from a row's id/notes fields, if present."""
    text = " ".join([row.get("id", ""), row.get("notes", "")])
    match = ARRIVE_RE.search(text)
    return match.group(0) if match else None


def get_start_timestamp(path: Path) -> int:
    """Return the timestamp_ms of the first row in a raw export file, used as the run's time origin."""
    with path.open(newline="") as f:
        reader = csv.DictReader(f)
        first = next(reader)
        return int(first["timestamp_ms"])


def generate_run(run_id: str, raw_file: str) -> tuple[int, int]:
    """Clean one run's raw export into separate BLE and IMU rows and store them in the global in-memory tables."""
    raw_path = RAW_DIR / raw_file
    out_dir = PROCESSED_DIR / run_id
    out_dir.mkdir(parents=True, exist_ok=True)
    start_ms = get_start_timestamp(raw_path)

    ble_path = out_dir / "ble_clean.csv"
    imu_path = out_dir / "imu_clean.csv"

    ble_fields = [
        "run_id",
        "time_sec",
        "timestamp_ms",
        "timestamp_iso",
        "ble_source",
        "beacon_id",
        "address",
        "rssi",
    ]
    imu_fields = [
        "run_id",
        "time_sec",
        "timestamp_ms",
        "timestamp_iso",
        "sensor",
        "x",
        "y",
        "z",
    ]

    seen_ble_rows: set[tuple[object, ...]] = set()
    seen_imu_rows: set[tuple[object, ...]] = set()
    ble_rows: list[dict[str, object]] = []
    imu_rows: list[dict[str, object]] = []

    with raw_path.open(newline="") as src:
        reader = csv.DictReader(src)
        for row in reader:
            timestamp_ms = int(row["timestamp_ms"])
            time_sec = round((timestamp_ms - start_ms) / 1000.0, 3)
            source = row.get("source", "")

            if source in BLE_SOURCES:
                beacon_id = extract_arrive_beacon_id(row)
                rssi = parse_float(row.get("rssi", ""))
                if time_sec <= STARTUP_EMPTY_ID_TRIM_SEC and not row.get("id", "").strip():
                    continue
                # rssi > 0 is not physically valid for BLE, and 127 is the Android "no reading" sentinel.
                if beacon_id is None or rssi is None or rssi > 0 or rssi == 127:
                    continue
                out_row = {
                    "run_id": run_id,
                    "time_sec": f"{time_sec:.3f}",
                    "timestamp_ms": timestamp_ms,
                    "timestamp_iso": row["timestamp_iso"],
                    "ble_source": source,
                    "beacon_id": beacon_id,
                    "address": row["address"],
                    "rssi": f"{rssi:.1f}",
                }
                row_key = tuple(out_row[field] for field in ble_fields)
                if row_key in seen_ble_rows:
                    continue
                seen_ble_rows.add(row_key)
                ble_rows.append(out_row)

            if source == "imu" and row.get("id") in IMU_SENSORS:
                x = parse_float(row.get("x", ""))
                y = parse_float(row.get("y", ""))
                z = parse_float(row.get("z", ""))
                if x is None or y is None or z is None:
                    continue
                out_row = {
                    "run_id": run_id,
                    "time_sec": f"{time_sec:.3f}",
                    "timestamp_ms": timestamp_ms,
                    "timestamp_iso": row["timestamp_iso"],
                    "sensor": row["id"],
                    "x": f"{x:.6f}",
                    "y": f"{y:.6f}",
                    "z": f"{z:.6f}",
                }
                row_key = tuple(out_row[field] for field in imu_fields)
                if row_key in seen_imu_rows:
                    continue
                seen_imu_rows.add(row_key)
                imu_rows.append(out_row)

    ble_rows.sort(key=lambda item: (float(item["time_sec"]), int(item["timestamp_ms"]), str(item["ble_source"])))
    imu_rows.sort(key=lambda item: (float(item["time_sec"]), int(item["timestamp_ms"]), str(item["sensor"])))

    ble_frame = pd.DataFrame(ble_rows, columns=ble_fields)
    imu_frame = pd.DataFrame(imu_rows, columns=imu_fields)

    # Keep the in-memory tables used by the next notebook section.
    GLOBAL_BLE_DATA[run_id] = ble_frame
    GLOBAL_IMU_DATA[run_id] = imu_frame

    # Also persist the generated intermediate files. Later notebook sections
    # intentionally reload these files, so no pre-existing processed data is
    # required in a fresh submission folder.
    ble_frame.to_csv(ble_path, index=False)
    imu_frame.to_csv(imu_path, index=False)

    return len(ble_rows), len(imu_rows)


def main() -> None:
    """Generate cleaned BLE/IMU data for every configured run and print a per-run row-count summary."""
    print("run_id,ble_rows,imu_rows")
    for run_id, raw_file in RUN_FILES.items():
        ble_count, imu_count = generate_run(run_id, raw_file)
        print(f"{run_id},{ble_count},{imu_count}")


if __name__ == "__main__":
    main()


Each row in the raw CSV represents one IMU or BLE event and is tagged with a `run_id`. After loading, `GLOBAL_BLE_DATA[run_id]` contains the accepted BLE observations (`time_sec`, `beacon_id`, `address`, and `rssi`). `GLOBAL_IMU_DATA[run_id]` contains the accelerometer, gyroscope, and orientation samples (`time_sec`, `sensor`, `x`, `y`, and `z`).


## 8. Data Preprocessing and Synchronization

Sensor times are measured relative to the first raw CSV row. Checkpoint matching subtracts the first cleaned IMU timestamp from PF time. These are distinct time conventions. BLE cleaning rejects unrecognized beacon IDs, missing/positive RSSI and sentinel 127; IMU cleaning requires numeric axes. Duplicate rows are removed. The code does not resample all streams to one common rate.


## 9. Exploratory Data Analysis

The cleaned IMU data shows the expected walking pattern. The acceleration magnitude contains a slowly changing gravity and phone-orientation component, with faster repeated peaks produced by footsteps. The gyroscope stays closer to zero on straight sections and changes more strongly during turns.

To isolate the walking motion, we smooth the acceleration magnitude over a short window and subtract a longer moving-average baseline. The remaining oscillations are much clearer and can be detected using local peaks, which forms the basis of the step detector in Section 10.

*(Plots of the raw and processed accelerometer, gyroscope, and smoothing result can be added here after running the notebook with the complete dataset.)*


## 10. Step Detection and Motion Model

**Step detection.** We detect a step when the de-trended acceleration magnitude forms a local peak above an adaptive threshold:

$$\text{threshold} = \max\Big(0.60,\; \bar{d} + 0.80\,\sigma_d\Big)$$

Here, $\bar d$ and $\sigma_d$ are the mean and standard deviation of the de-trended signal. A minimum gap of 0.35 s prevents one physical step from being counted more than once.

**Step-length estimation.** We estimate the length of each step using Weinberg's method:

$$\hat{L} = K \cdot \big(a_{\max} - a_{\min}\big)^{1/4}, \qquad K = 0.70$$

The acceleration range is taken from the window around the detected step.

**Generalized heading estimation.** The absolute phone yaw is not assumed to be aligned with map north. A causal circular window first smooths `imu_processed.x`. The heading change is then represented relative to the first filtered yaw:

$$\Delta\theta_k = \operatorname{wrap}_{180}(\theta_k-\theta_{k-1}), \qquad \theta_k^{rel}=\operatorname{wrap}_{360}(\theta_k-\theta_0)$$

This cancels a constant phone-to-map offset. During large turns, the gyroscope is projected onto the estimated gravity direction and integrated using the recorded timestamps. A turn is accepted only when both orientation and gyro data support it. The gyro correction remains active after the turn instead of being applied to one isolated step.

**PDR motion model.** The relative position is updated after every detected step:

$$x_{k} = x_{k-1} + \hat L_k \cos(\theta_k), \qquad y_{k} = y_{k-1} + \hat L_k \sin(\theta_k)$$

The basic PDR plot remains in the phone's sensor frame. The particle filter later maps the relative heading to the floor plan by maintaining several map-derived initial direction hypotheses around the known start. No run-specific heading offsets or reference-route directions are used.

The implementation below detects the steps, estimates their lengths and relative headings, attaches the available BLE context, and generates a PDR trajectory for each run. This is a step-and-heading PDR approach of the type reviewed by Harle [1].

In [ ]:
from __future__ import annotations
"""Generate step detections and preliminary position estimates per run."""

from pathlib import Path
from collections import defaultdict
import math
import statistics
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression

ROOT = Path.cwd()
PROCESSED_DIR = ROOT / "data" / "processed"
METADATA_DIR = ROOT / "data" / "metadata"
RUN_IDS = ("R1","R2","R3","R4")

DEFAULT_STEP_LENGTH_M = 0.55
WEINBERG_K = 0.70
MIN_STEP_GAP_SEC = 0.35
BLE_CONTEXT_WINDOW_SEC = 1.5
HEADING_OUTLIER_DEG = 80.0
HEADING_SMOOTH_WINDOW_SEC = 0.60
GYRO_GRAVITY_TAU_SEC = 0.75
GYRO_TURN_WINDOW_STEPS = 4
POST_TURN_STABILIZE_STEPS = 3
STEP_LENGTH_TRAINING_FILE = ROOT / "data" / "calibration" / "step_length_training.csv"
STEP_REGRESSION_BLEND = 0.25
STEP_REGRESSION_ALPHA = 10.0
STEP_LENGTH_MIN_M = 0.35
STEP_LENGTH_MAX_M = 1.35
STEP_MODEL_FEATURES = [
    "weinberg_feature", "cadence_hz", "dynamic_peak",
    "abs_gyro_delta", "abs_orientation_delta",
    "gyro_activity", "dynamic_activity",
]
_STEP_LENGTH_MODEL = None


def moving_average(values, window):
    """Compute a simple centered moving average of a list of values over the given window size."""
    if not values:
        return []
    window = max(1, min(window, len(values)))
    half, prefix = window // 2, [0.0]
    for value in values:
        prefix.append(prefix[-1] + value)
    return [
        (prefix[min(len(values), i + half + 1)] - prefix[max(0, i - half)])
        / (min(len(values), i + half + 1) - max(0, i - half))
        for i in range(len(values))
    ]


def read_imu(run_id):
    """Load a run's cleaned IMU rows and group them by sensor (accel/gyro/imu_processed), sorted by time."""
    df = GLOBAL_IMU_DATA.get(run_id, pd.DataFrame()).copy()
    # Cleaned rows are initially created from formatted CSV strings. Convert
    # time and sensor values back to numeric before sorting and signal analysis.
    numeric_columns = ["time_sec", "timestamp_ms", "x", "y", "z"]
    df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=["time_sec", "x", "y", "z"])
    df["magnitude"] = (df.x**2 + df.y**2 + df.z**2) ** 0.5
    return {
        sensor: group.sort_values("time_sec").to_dict("records")
        for sensor, group in df.groupby("sensor")
    }


def read_ble(run_id):
    """Load a run's cleaned, de-duplicated BLE rows sorted by time."""
    df = GLOBAL_BLE_DATA.get(run_id, pd.DataFrame()).copy()
    df[["time_sec", "timestamp_ms", "rssi"]] = df[
        ["time_sec", "timestamp_ms", "rssi"]
    ].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=["time_sec", "rssi"])
    return (
        df.drop_duplicates(subset=["time_sec", "beacon_id", "address", "rssi"])
        .sort_values("time_sec")
        .to_dict("records")
    )


def read_beacon_locations():
    """Load beacon x/y coordinates (and floor) from the beacon metadata CSV, keyed by beacon_id."""
    df = pd.read_csv(METADATA_DIR / "beacon_locations.csv")
    df[["x", "y"]] = df[["x", "y"]].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=["x", "y"])
    return {
        row.beacon_id: (row.x, row.y, getattr(row, "floor", ""))
        for row in df.itertuples()
    }


def angle_difference(a, b):
    """Return the signed angular difference a-b, wrapped to [-180, 180]."""
    return (float(a) - float(b) + 180.0) % 360.0 - 180.0


def nearest_heading(orientation, time_sec):
    """Return the most recent available orientation yaw at or before a step."""
    if not orientation:
        return None
    valid = [row for row in orientation if float(row["time_sec"]) <= float(time_sec)]
    row = valid[-1] if valid else orientation[0]
    return float(row["x"]) % 360.0


def circular_mean_deg(values):
    """Return the circular mean of degree-valued angles."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return float("nan")
    radians = np.deg2rad(values)
    return float(np.rad2deg(np.arctan2(np.sin(radians).mean(), np.cos(radians).mean())) % 360.0)


def robust_mad(values):
    """Median absolute deviation for finite values."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0.0
    median = float(np.median(values))
    return float(np.median(np.abs(values - median)))


def causal_orientation_at_steps(orientation, step_times):
    """Estimate a robust yaw at every step using only recent orientation samples."""
    if not orientation:
        return np.full(len(step_times), np.nan)

    frame = pd.DataFrame(orientation).copy()
    frame[["time_sec", "x"]] = frame[["time_sec", "x"]].apply(pd.to_numeric, errors="coerce")
    frame = frame.dropna(subset=["time_sec", "x"]).sort_values("time_sec").drop_duplicates("time_sec")
    if frame.empty:
        return np.full(len(step_times), np.nan)

    times = frame["time_sec"].to_numpy(dtype=float)
    yaw = np.mod(frame["x"].to_numpy(dtype=float), 360.0)
    estimates = []
    previous = float(yaw[0])

    for step_time in np.asarray(step_times, dtype=float):
        mask = (times <= step_time) & (times >= step_time - HEADING_SMOOTH_WINDOW_SEC)
        values = yaw[mask]
        if values.size == 0:
            index = int(np.searchsorted(times, step_time, side="right") - 1)
            values = yaw[max(0, index):max(0, index) + 1]
        estimate = circular_mean_deg(values)
        if not np.isfinite(estimate):
            estimate = previous
        estimates.append(estimate)
        previous = estimate

    return np.asarray(estimates, dtype=float)


def low_pass_irregular(times, vectors, tau_sec):
    """First-order low-pass filter for an irregularly sampled vector stream."""
    times = np.asarray(times, dtype=float)
    vectors = np.asarray(vectors, dtype=float)
    filtered = np.empty_like(vectors)
    filtered[0] = vectors[0]
    for index in range(1, len(vectors)):
        dt = max(float(times[index] - times[index - 1]), 1e-4)
        alpha = dt / (tau_sec + dt)
        filtered[index] = filtered[index - 1] + alpha * (vectors[index] - filtered[index - 1])
    return filtered


def integrate_interval(times, values, start_sec, end_sec):
    """Integrate a signal over one step interval using its real timestamps."""
    if end_sec <= start_sec or len(times) < 2:
        return 0.0
    inside = (times > start_sec) & (times < end_sec)
    interval_times = np.concatenate(([start_sec], times[inside], [end_sec]))
    interval_values = np.interp(interval_times, times, values)
    return float(np.trapezoid(interval_values, interval_times))


def gyro_rotation_per_step(accel, gyro, step_times):
    """Project angular velocity onto gravity and integrate it between step events."""
    if not accel or not gyro or len(step_times) == 0:
        return np.zeros(len(step_times), dtype=float)

    accel_frame = pd.DataFrame(accel).copy()
    gyro_frame = pd.DataFrame(gyro).copy()
    numeric = ["time_sec", "x", "y", "z"]
    accel_frame[numeric] = accel_frame[numeric].apply(pd.to_numeric, errors="coerce")
    gyro_frame[numeric] = gyro_frame[numeric].apply(pd.to_numeric, errors="coerce")
    accel_frame = accel_frame.dropna(subset=numeric).sort_values("time_sec").drop_duplicates("time_sec")
    gyro_frame = gyro_frame.dropna(subset=numeric).sort_values("time_sec").drop_duplicates("time_sec")
    if len(accel_frame) < 2 or len(gyro_frame) < 2:
        return np.zeros(len(step_times), dtype=float)

    accel_times = accel_frame["time_sec"].to_numpy(dtype=float)
    accel_values = accel_frame[["x", "y", "z"]].to_numpy(dtype=float)
    gravity = low_pass_irregular(accel_times, accel_values, GYRO_GRAVITY_TAU_SEC)
    gravity /= np.maximum(np.linalg.norm(gravity, axis=1, keepdims=True), 1e-9)

    gyro_times = gyro_frame["time_sec"].to_numpy(dtype=float)
    gyro_values = gyro_frame[["x", "y", "z"]].to_numpy(dtype=float)
    gravity_at_gyro = np.column_stack([
        np.interp(gyro_times, accel_times, gravity[:, axis])
        for axis in range(3)
    ])
    gravity_at_gyro /= np.maximum(np.linalg.norm(gravity_at_gyro, axis=1, keepdims=True), 1e-9)

    vertical_rate = np.sum(gyro_values * gravity_at_gyro, axis=1)
    gyro_norm = np.linalg.norm(gyro_values, axis=1)
    quiet = gyro_norm <= np.quantile(gyro_norm, 0.25)
    gyro_bias = float(np.median(vertical_rate[quiet])) if quiet.any() else 0.0
    corrected_rate = vertical_rate - gyro_bias

    step_times = np.asarray(step_times, dtype=float)
    previous_times = np.concatenate(([max(0.0, step_times[0] - 0.5)], step_times[:-1]))
    return np.asarray([
        math.degrees(integrate_interval(gyro_times, corrected_rate, start, end))
        for start, end in zip(previous_times, step_times)
    ], dtype=float)


def adaptive_turn_probabilities(orientation_delta, gyro_delta):
    """Estimate turn probability with a small logistic model fitted per recording.

    Clear straight and clear turning intervals are selected from the recording's
    own gyro and orientation quantiles. These high-confidence intervals become
    pseudo-labels for a regularized logistic model. The probability is retained
    as a diagnostic and uncertainty signal, while the physically validated gyro
    fusion remains responsible for the actual heading update.
    """
    orientation_delta = np.asarray(orientation_delta, dtype=float)
    gyro_delta = np.asarray(gyro_delta, dtype=float)
    rolling_gyro = pd.Series(gyro_delta).rolling(
        GYRO_TURN_WINDOW_STEPS, min_periods=1
    ).sum().to_numpy()
    rolling_orientation = pd.Series(orientation_delta).rolling(
        GYRO_TURN_WINDOW_STEPS, min_periods=1
    ).sum().to_numpy()
    agreement = np.sign(gyro_delta * orientation_delta)
    features = np.column_stack((
        np.abs(gyro_delta), np.abs(orientation_delta),
        np.abs(rolling_gyro), np.abs(rolling_orientation), agreement,
    ))

    gyro_high = max(25.0, float(np.quantile(np.abs(rolling_gyro), 0.85)))
    orientation_high = max(20.0, float(np.quantile(np.abs(rolling_orientation), 0.80)))
    gyro_low = float(np.quantile(np.abs(rolling_gyro), 0.55))
    orientation_low = float(np.quantile(np.abs(rolling_orientation), 0.55))

    positive = (
        (np.abs(rolling_gyro) >= gyro_high)
        & (np.abs(rolling_orientation) >= orientation_high)
        & (agreement >= 0)
    )
    negative = (
        (np.abs(rolling_gyro) <= gyro_low)
        & (np.abs(rolling_orientation) <= orientation_low)
    )
    labels = np.full(len(features), -1, dtype=int)
    labels[negative] = 0
    labels[positive] = 1
    training_mask = labels >= 0

    if positive.sum() < 2 or negative.sum() < 8:
        score = np.maximum(
            np.abs(rolling_gyro) / max(gyro_high, 1e-6),
            np.abs(rolling_orientation) / max(orientation_high, 1e-6),
        )
        return np.clip(score, 0.0, 1.0), "adaptive_rule_fallback"

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=1.0, class_weight="balanced", max_iter=1000, random_state=0
        ),
    )
    model.fit(features[training_mask], labels[training_mask])
    return model.predict_proba(features)[:, 1], "per_run_logistic"


def fit_step_length_regressor():
    """Fit the cross-session Ridge model used as a conservative step prior.

    The calibration table comes from the separate old recording session. The
    current run's reference points are never used to fit or update this model.
    """
    global _STEP_LENGTH_MODEL
    if _STEP_LENGTH_MODEL is not None:
        return _STEP_LENGTH_MODEL
    if not STEP_LENGTH_TRAINING_FILE.exists():
        return None
    training = pd.read_csv(STEP_LENGTH_TRAINING_FILE).dropna(
        subset=STEP_MODEL_FEATURES + ["target_horizontal_step_m"]
    )
    if len(training) < 20:
        return None
    model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=STEP_REGRESSION_ALPHA),
    )
    model.fit(
        training[STEP_MODEL_FEATURES],
        training["target_horizontal_step_m"],
    )
    _STEP_LENGTH_MODEL = model
    return model


def step_model_features(steps):
    """Build event-level features for the adaptive step-length model."""
    frame = pd.DataFrame(steps).copy()
    if frame.empty:
        return pd.DataFrame(columns=STEP_MODEL_FEATURES)
    for column in [
        "time_sec", "step_length_m", "dynamic_accel_peak",
        "gyro_delta_deg", "orientation_delta_deg",
    ]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    interval = frame["time_sec"].diff()
    median_interval = float(interval.dropna().median()) if interval.notna().any() else 0.55
    interval = interval.fillna(median_interval).clip(0.15, 2.0)
    features = pd.DataFrame({
        "weinberg_feature": frame["step_length_m"] / WEINBERG_K,
        "cadence_hz": 1.0 / interval,
        "dynamic_peak": frame["dynamic_accel_peak"].fillna(
            frame["dynamic_accel_peak"].median()
        ),
        "abs_gyro_delta": frame["gyro_delta_deg"].abs().fillna(0.0),
        "abs_orientation_delta": frame["orientation_delta_deg"].abs().fillna(0.0),
    })
    features["gyro_activity"] = features["abs_gyro_delta"].rolling(
        5, min_periods=1, center=True
    ).mean()
    features["dynamic_activity"] = features["dynamic_peak"].rolling(
        5, min_periods=1, center=True
    ).mean()
    return features[STEP_MODEL_FEATURES]


def apply_adaptive_step_length(steps):
    """Blend Weinberg distance with a cross-session Ridge prediction.

    A conservative blend is used because the old and final sessions do not have
    perfectly identical movement protocols. This keeps the physical Weinberg
    estimate active while allowing cadence, acceleration, and turn activity to
    adapt the distance for the current recording.
    """
    if not steps:
        return
    model = fit_step_length_regressor()
    features = step_model_features(steps)
    if model is None or features.empty:
        for step in steps:
            step["step_length_weinberg_m"] = float(step["step_length_m"])
            step["step_length_regression_m"] = float(step["step_length_m"])
            step["step_length_model"] = "weinberg_fallback"
        return

    prediction = np.clip(
        model.predict(features), STEP_LENGTH_MIN_M, STEP_LENGTH_MAX_M
    )
    for index, step in enumerate(steps):
        weinberg = float(step["step_length_m"])
        adaptive = float(prediction[index])
        blended = (1.0 - STEP_REGRESSION_BLEND) * weinberg + STEP_REGRESSION_BLEND * adaptive
        step["step_length_weinberg_m"] = weinberg
        step["step_length_regression_m"] = adaptive
        step["step_length_m"] = float(np.clip(
            blended, STEP_LENGTH_MIN_M, STEP_LENGTH_MAX_M
        ))
        step["step_length_model"] = "weinberg_ridge_blend"


def apply_generalized_heading(steps, orientation, gyro, accel):
    """Create a session-independent heading using relative yaw and gyro turns.

    The first filtered phone yaw is treated only as the origin of the phone frame.
    Constant phone-to-map offsets therefore cancel. Causal orientation supplies the
    normal relative change, while timestamp-integrated gyro rotation replaces only
    large, sensor-confirmed turn intervals and remains persistent after the turn.
    """
    if not steps:
        return

    step_times = np.asarray([float(step["time_sec"]) for step in steps], dtype=float)
    sensor_heading = causal_orientation_at_steps(orientation, step_times)
    if not np.isfinite(sensor_heading).any():
        sensor_heading = np.zeros(len(steps), dtype=float)
    else:
        first_valid = sensor_heading[np.isfinite(sensor_heading)][0]
        sensor_heading = np.where(np.isfinite(sensor_heading), sensor_heading, first_valid)

    orientation_delta = np.zeros(len(steps), dtype=float)
    base_relative = np.zeros(len(steps), dtype=float)
    for index in range(1, len(steps)):
        orientation_delta[index] = angle_difference(sensor_heading[index], sensor_heading[index - 1])
        base_relative[index] = angle_difference(sensor_heading[index], sensor_heading[0])

    raw_gyro_delta = gyro_rotation_per_step(accel, gyro, step_times)
    sign_mask = (np.abs(raw_gyro_delta) >= 2.0) & (np.abs(orientation_delta) >= 2.0)
    sign_agreement = np.sum(raw_gyro_delta[sign_mask] * orientation_delta[sign_mask])
    gyro_sign = 1.0 if sign_agreement >= 0.0 else -1.0
    gyro_delta = gyro_sign * raw_gyro_delta
    turn_probability, turn_model = adaptive_turn_probabilities(
        orientation_delta, gyro_delta
    )

    abs_gyro = np.abs(gyro_delta)
    gyro_noise = float(np.median(abs_gyro) + 3.0 * 1.4826 * robust_mad(abs_gyro))
    step_threshold = max(10.0, gyro_noise)
    rolling_gyro = pd.Series(gyro_delta).rolling(GYRO_TURN_WINDOW_STEPS, min_periods=1).sum().to_numpy()
    rolling_orientation = pd.Series(orientation_delta).rolling(GYRO_TURN_WINDOW_STEPS, min_periods=1).sum().to_numpy()

    candidate = (
        (np.abs(rolling_gyro) >= max(40.0, 2.5 * step_threshold))
        & (np.abs(rolling_orientation) >= 30.0)
    ) | (
        (abs_gyro >= max(25.0, 2.0 * step_threshold))
        & (np.abs(orientation_delta) >= 20.0)
    )

    candidate_indices = np.flatnonzero(candidate)
    clusters = []
    if len(candidate_indices):
        cluster_start = previous = int(candidate_indices[0])
        for index in candidate_indices[1:]:
            index = int(index)
            if index > previous + 1:
                clusters.append((max(1, cluster_start - 1), min(len(steps) - 1, previous + 1)))
                cluster_start = index
            previous = index
        clusters.append((max(1, cluster_start - 1), min(len(steps) - 1, previous + 1)))

    accepted = []
    for cluster_start, cluster_end in clusters:
        gyro_rotation = float(gyro_delta[cluster_start:cluster_end + 1].sum())
        orientation_rotation = float(angle_difference(
            sensor_heading[cluster_end], sensor_heading[max(0, cluster_start - 1)]
        ))
        if abs(gyro_rotation) >= 55.0 and abs(orientation_rotation) >= 40.0:
            if not accepted or cluster_start > accepted[-1][1] + 1:
                accepted.append((cluster_start, cluster_end))
            else:
                accepted[-1] = (accepted[-1][0], max(accepted[-1][1], cluster_end))

    fused_relative = base_relative.copy()
    heading_mode = np.full(len(steps), "relative_orientation", dtype=object)
    persistent_correction = 0.0
    cluster_by_start = {start: end for start, end in accepted}
    index = 1

    while index < len(steps):
        if index in cluster_by_start:
            cluster_end = cluster_by_start[index]
            fused_relative[index] = fused_relative[index - 1] + gyro_delta[index]
            heading_mode[index] = "gyro_turn"
            for turn_index in range(index + 1, cluster_end + 1):
                fused_relative[turn_index] = fused_relative[turn_index - 1] + gyro_delta[turn_index]
                heading_mode[turn_index] = "gyro_turn"
            persistent_correction = angle_difference(
                fused_relative[cluster_end], base_relative[cluster_end]
            )
            index = cluster_end + 1
            continue

        # Keep the gyro correction after a turn while following later relative yaw.
        target = base_relative[index] + persistent_correction
        change = angle_difference(target, fused_relative[index - 1])
        # Reject a large one-event orientation jump when the gyro does not support it.
        if abs(change) > 55.0 and abs(gyro_delta[index]) < step_threshold:
            fused_relative[index] = fused_relative[index - 1]
            heading_mode[index] = "magnetic_rejection"
        else:
            fused_relative[index] = target
            heading_mode[index] = "relative_orientation"
        index += 1

    fused_relative = np.mod(fused_relative, 360.0)
    fused_sensor = np.mod(sensor_heading[0] + fused_relative, 360.0)

    for index, step in enumerate(steps):
        step["heading_deg_sensor"] = float(sensor_heading[index])
        step["heading_deg"] = float(fused_sensor[index])
        step["heading_relative_deg"] = float(fused_relative[index])
        step["orientation_delta_deg"] = float(orientation_delta[index])
        step["gyro_delta_deg"] = float(gyro_delta[index])
        step["heading_mode"] = str(heading_mode[index])
        step["turn_probability"] = float(turn_probability[index])
        step["turn_model"] = str(turn_model)

def detect_steps(run_id, streams):
    """Detect footsteps from the accelerometer magnitude signal and estimate each step's length, heading, and BLE context."""
    accel = streams.get("accel", [])
    if len(accel) < 20:
        return []

    times = [float(row["time_sec"]) for row in accel]
    magnitudes = [float(row["magnitude"]) for row in accel]
    sample_rate = len(accel) / max(times[-1] - times[0], 1e-6)

    smooth = moving_average(magnitudes, max(3, int(sample_rate * 0.08)))
    baseline = moving_average(smooth, max(5, int(sample_rate * 0.75)))
    dynamic = [value - base for value, base in zip(smooth, baseline)]

        # Adaptive threshold: a run-specific floor (0.60) plus a multiple of the
        # signal's own mean/std, so noisier or calmer runs get different sensitivity.
    threshold = max(0.60, statistics.fmean(dynamic) + 0.80 * statistics.pstdev(dynamic))
    min_gap = max(1, int(sample_rate * MIN_STEP_GAP_SEC))
    steps, last_peak, last_step = [], -min_gap, 0

    for i in range(1, len(dynamic) - 1):
        is_peak = (
            i - last_peak >= min_gap
            and dynamic[i] > threshold
            and dynamic[i] >= dynamic[i - 1]
            and dynamic[i] >= dynamic[i + 1]
        )
        if not is_peak:
            continue

        window = magnitudes[last_step:i + 1]
                # Weinberg's model: step length scales with the 4th root of the
                # peak-to-peak acceleration range within the step window.
        step_length = (
            WEINBERG_K * max(0.0, max(window) - min(window)) ** 0.25
            if window else DEFAULT_STEP_LENGTH_M
        )
        row = accel[i]
        steps.append({
            "run_id": run_id,
            "step_id": len(steps) + 1,
            "time_sec": float(row["time_sec"]),
            "timestamp_ms": int(row["timestamp_ms"]),
            "timestamp_iso": row["timestamp_iso"],
            "accel_magnitude": float(row["magnitude"]),
            "dynamic_accel_peak": dynamic[i],
            "step_length_m": step_length,
            "heading_deg": nearest_heading(streams.get("imu_processed", []), float(row["time_sec"])),
            "method": "weinberg",
        })
        last_peak = last_step = i
    apply_generalized_heading(
        steps,
        streams.get("imu_processed", []),
        streams.get("gyro", []),
        streams.get("accel", []),
    )
    apply_adaptive_step_length(steps)

    return steps


def empty_ble_context():
    """Return a placeholder BLE-context dictionary used when no beacon observations are available near a step."""
    return {
        "dominant_beacon_id": "", "dominant_rssi": "",
        "second_beacon_id": "", "second_rssi": "",
        "ble_observation_count": 0, "ble_confidence": "none",
        "ble_weighted_x": "", "ble_weighted_y": "", "ble_floor": "",
    }


def ble_context(ble_rows, beacon_locations, time_sec):
    """Summarize nearby BLE observations around a given time into a dominant/second beacon, confidence level, and RSSI-weighted position."""
    grouped = defaultdict(list)
    half_window = BLE_CONTEXT_WINDOW_SEC / 2
    for row in ble_rows:
        if abs(float(row["time_sec"]) - time_sec) <= half_window:
            grouped[str(row["beacon_id"])].append(float(row["rssi"]))
    if not grouped:
        return empty_ble_context()

    ranked = sorted(
        ((beacon, max(values), len(values)) for beacon, values in grouped.items()),
        key=lambda item: item[1], reverse=True,
    )
    dominant = ranked[0]
    second = ranked[1] if len(ranked) > 1 else ("", "", 0)
    confidence = "strong" if dominant[1] >= -65 else "medium" if dominant[1] >= -78 else "weak"

    points = []
    for beacon, values in grouped.items():
        if beacon in beacon_locations:
            x, y, floor = beacon_locations[beacon]
            points.append((x, y, floor, 10 ** (max(values) / 20.0)))

    weighted_x = weighted_y = floor = ""
    if points:
        total = sum(weight for *_, weight in points)
        weighted_x = sum(x * weight for x, _, _, weight in points) / total
        weighted_y = sum(y * weight for _, y, _, weight in points) / total
        floor_weights = defaultdict(float)
        for _, _, point_floor, weight in points:
            floor_weights[point_floor] += weight
        floor = max(floor_weights, key=floor_weights.get)

    return {
        "dominant_beacon_id": dominant[0], "dominant_rssi": dominant[1],
        "second_beacon_id": second[0], "second_rssi": second[1],
        "ble_observation_count": sum(map(len, grouped.values())),
        "ble_confidence": confidence,
        "ble_weighted_x": weighted_x, "ble_weighted_y": weighted_y,
        "ble_floor": floor,
    }


def fmt(value, digits):
    """Format a numeric value with a fixed number of decimals, or return an empty string for missing values."""
    return "" if value == "" or value is None else f"{float(value):.{digits}f}"


def write_steps(run_id, steps):
    """Format detected steps into a DataFrame and store it in the global steps table for the given run."""
    rows = []
    for step in steps:
        row = step.copy()
        row.update(
            time_sec=f"{row['time_sec']:.3f}",
            accel_magnitude=f"{row['accel_magnitude']:.6f}",
            dynamic_accel_peak=f"{row['dynamic_accel_peak']:.6f}",
            step_length_m=f"{row['step_length_m']:.3f}",
            step_length_weinberg_m=fmt(row.get("step_length_weinberg_m"), 3),
            step_length_regression_m=fmt(row.get("step_length_regression_m"), 3),
            heading_deg=fmt(row["heading_deg"], 2),
            heading_deg_sensor=fmt(row.get("heading_deg_sensor"), 2),
            heading_relative_deg=fmt(row.get("heading_relative_deg"), 2),
            orientation_delta_deg=fmt(row.get("orientation_delta_deg"), 2),
            gyro_delta_deg=fmt(row.get("gyro_delta_deg"), 2),
            turn_probability=fmt(row.get("turn_probability"), 3),
        )
        rows.append(row)

    columns = [
        "run_id", "step_id", "time_sec", "timestamp_ms", "timestamp_iso",
        "accel_magnitude", "dynamic_accel_peak", "step_length_m",
        "step_length_weinberg_m", "step_length_regression_m", "step_length_model",
        "heading_deg", "heading_deg_sensor", "heading_relative_deg",
        "orientation_delta_deg", "gyro_delta_deg", "turn_probability",
        "turn_model", "heading_mode", "method",
    ]
    frame = pd.DataFrame(rows, columns=columns)
    GLOBAL_STEPS_DATA[run_id] = frame

    # Persist the detected steps for the particle-filter section. The folder
    # is created on demand so the notebook works without bundled processed data.
    out_dir = PROCESSED_DIR / run_id
    out_dir.mkdir(parents=True, exist_ok=True)
    frame.to_csv(out_dir / "steps.csv", index=False)


def write_positions(run_id, steps, ble_rows, beacon_locations):
    """Accumulate step-by-step PDR displacements (with attached BLE context) into a DataFrame and store it in the global PDR table for the given run."""
    x_rel = y_rel = 0.0
    rows = []

    for estimate_id, step in enumerate(steps, 1):
        heading, length = step["heading_deg"], float(step["step_length_m"])
        if heading is not None:
            angle = math.radians(float(heading))
            x_rel += length * math.cos(angle)
            y_rel += length * math.sin(angle)

        context = ble_context(ble_rows, beacon_locations, float(step["time_sec"]))
        rows.append({
            "run_id": run_id, "estimate_id": estimate_id,
            "time_sec": f"{float(step['time_sec']):.3f}",
            "timestamp_ms": step["timestamp_ms"], "timestamp_iso": step["timestamp_iso"],
            "x_rel_m": f"{x_rel:.3f}", "y_rel_m": f"{y_rel:.3f}",
            "heading_deg": fmt(heading, 2), "step_id": step["step_id"],
            "step_length_m": f"{length:.2f}",
            "dominant_beacon_id": context["dominant_beacon_id"],
            "dominant_rssi": fmt(context["dominant_rssi"], 1),
            "second_beacon_id": context["second_beacon_id"],
            "second_rssi": fmt(context["second_rssi"], 1),
            "ble_observation_count": context["ble_observation_count"],
            "ble_confidence": context["ble_confidence"],
            "ble_weighted_x": fmt(context["ble_weighted_x"], 3),
            "ble_weighted_y": fmt(context["ble_weighted_y"], 3),
            "ble_floor": context["ble_floor"],
            "method": "step_dead_reckoning_with_ble_context",
        })

    columns = [
        "run_id", "estimate_id", "time_sec", "timestamp_ms", "timestamp_iso",
        "x_rel_m", "y_rel_m", "heading_deg", "step_id", "step_length_m",
        "dominant_beacon_id", "dominant_rssi", "second_beacon_id", "second_rssi",
        "ble_observation_count", "ble_confidence", "ble_weighted_x",
        "ble_weighted_y", "ble_floor", "method",
    ]
    frame = pd.DataFrame(rows, columns=columns)
    GLOBAL_PDR_DATA[run_id] = frame

    # Save the preliminary PDR trajectory as a reproducible intermediate result.
    out_dir = PROCESSED_DIR / run_id
    out_dir.mkdir(parents=True, exist_ok=True)
    frame.to_csv(out_dir / "position_estimates.csv", index=False)


def main():
    """Run step detection and PDR position estimation for every configured run and print a per-run summary."""
    beacon_locations = read_beacon_locations()
    print("run_id,steps,position_estimates,beacon_locations_with_xy")

    for run_id in RUN_IDS:
        streams = read_imu(run_id)
        ble_rows = read_ble(run_id)
        steps = detect_steps(run_id, streams)
        write_steps(run_id, steps)
        write_positions(run_id, steps, ble_rows, beacon_locations)
        print(f"{run_id},{len(steps)},{len(steps)},{len(beacon_locations)}")


if __name__ == "__main__":
    main()



### 10.1 Adaptive Regression Support

The final motion model keeps the physical Weinberg estimate, but adds two lightweight regression components. A Ridge regression model predicts horizontal movement distance from cadence, acceleration range, gyro activity, and orientation change. It is trained only on the separate old recording session and is blended conservatively with Weinberg, so the current run's reference checkpoints are not used for online correction.

A small logistic regression is fitted independently for every recording to estimate turn probability. Its pseudo-labels come from clear straight and clear turning intervals identified from that run's own gyro and orientation distributions. The probability is stored for diagnostics. The actual heading still uses the relative-yaw and timestamp-integrated gyro logic, because controlled testing showed that directly replacing the physical turn detector with logistic decisions degraded R3 and R4.

The old recordings are calibration and stress-test data, not a second accuracy benchmark. Their checkpoint times and, in some cases, their protocols differ from the final runs. In particular, `R3_old` ends before the final run's staircase interval, so no floor transition should be expected there.


In [ ]:

# Adaptive model diagnostics for each final run
adaptive_rows = []
for run_id in RUN_IDS:
    step_frame = GLOBAL_STEPS_DATA.get(run_id, pd.DataFrame()).copy()
    if step_frame.empty:
        continue
    numeric = [
        "step_length_m", "step_length_weinberg_m",
        "step_length_regression_m", "turn_probability",
    ]
    for column in numeric:
        if column in step_frame:
            step_frame[column] = pd.to_numeric(step_frame[column], errors="coerce")
    adaptive_rows.append({
        "run_id": run_id,
        "events": len(step_frame),
        "mean_weinberg_m": step_frame["step_length_weinberg_m"].mean(),
        "mean_regression_m": step_frame["step_length_regression_m"].mean(),
        "mean_used_m": step_frame["step_length_m"].mean(),
        "maximum_turn_probability": step_frame["turn_probability"].max(),
        "turn_model": step_frame["turn_model"].iloc[0],
    })

adaptive_model_summary = pd.DataFrame(adaptive_rows)
display(adaptive_model_summary.round(3))


### 10.1 PDR and Step-Detection Visualizations

For each of the four runs, the plots below show:

1. acceleration magnitude and the accepted movement-event times,
2. estimated step length and heading over time, and
3. the accumulated PDR-only trajectory in metres.

These plots use only the IMU-based motion estimate. BLE measurements, floor-map constraints, and reference points are not included, so the trajectory shows the PDR behaviour before particle-filter correction.


In [ ]:
# PDR diagnostics for every run: sensor activity, event inputs, and PDR-only path.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def _numeric_copy(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Return a copy with selected columns converted to numeric values."""
    result = frame.copy()
    for column in columns:
        if column in result.columns:
            result[column] = pd.to_numeric(result[column], errors="coerce")
    return result


for run_id in RUN_IDS:
    imu = _numeric_copy(GLOBAL_IMU_DATA[run_id], ["time_sec", "x", "y", "z"])
    accel = imu[imu["sensor"].eq("accel")].dropna(subset=["time_sec", "x", "y", "z"]).copy()
    accel["magnitude"] = np.sqrt(accel["x"] ** 2 + accel["y"] ** 2 + accel["z"] ** 2)

    steps = _numeric_copy(
        GLOBAL_STEPS_DATA[run_id],
        ["time_sec", "step_length_m", "heading_deg", "accel_magnitude"],
    ).dropna(subset=["time_sec", "step_length_m", "heading_deg"])

    pdr = _numeric_copy(
        GLOBAL_PDR_DATA[run_id],
        ["time_sec", "x_rel_m", "y_rel_m"],
    ).dropna(subset=["x_rel_m", "y_rel_m"])

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
    fig.suptitle(f"{run_id}: PDR input and trajectory diagnostics", fontsize=14)

    # 1) Raw acceleration magnitude and accepted event times.
    axes[0].plot(accel["time_sec"], accel["magnitude"], linewidth=0.7, alpha=0.75, label="acceleration magnitude")
    if not steps.empty and not accel.empty:
        marker_y = np.interp(
            steps["time_sec"].to_numpy(),
            accel["time_sec"].to_numpy(),
            accel["magnitude"].to_numpy(),
        )
        axes[0].scatter(steps["time_sec"], marker_y, s=18, marker="x", label="detected movement event")
    axes[0].set_title("Acceleration and detected events")
    axes[0].set_xlabel("Movement-relative time, s")
    axes[0].set_ylabel("Acceleration magnitude, m/s²")
    axes[0].grid(True, alpha=0.25)
    axes[0].legend(fontsize=8)

    # 2) Distance and heading supplied to PDR/PF.
    distance_axis = axes[1]
    heading_axis = distance_axis.twinx()
    distance_axis.plot(steps["time_sec"], steps["step_length_m"], linewidth=1.2, label="step length")
    heading_axis.plot(steps["time_sec"], steps["heading_deg"], linewidth=1.0, alpha=0.75, label="heading")
    distance_axis.set_title("Event-wise PDR inputs")
    distance_axis.set_xlabel("Movement-relative time, s")
    distance_axis.set_ylabel("Estimated event distance, m")
    heading_axis.set_ylabel("Heading, degrees")
    heading_axis.set_ylim(0, 360)
    distance_axis.grid(True, alpha=0.25)
    lines = distance_axis.get_lines() + heading_axis.get_lines()
    distance_axis.legend(lines, [line.get_label() for line in lines], fontsize=8, loc="best")

    # 3) Relative PDR trajectory before BLE/map correction.
    axes[2].plot(pdr["x_rel_m"], pdr["y_rel_m"], linewidth=1.8, label="PDR-only path")
    if not pdr.empty:
        axes[2].scatter(pdr.iloc[0]["x_rel_m"], pdr.iloc[0]["y_rel_m"], s=75, label="start", zorder=3)
        axes[2].scatter(pdr.iloc[-1]["x_rel_m"], pdr.iloc[-1]["y_rel_m"], marker="X", s=85, label="end", zorder=3)
    axes[2].set_title("Accumulated PDR-only trajectory")
    axes[2].set_xlabel("Relative x, m")
    axes[2].set_ylabel("Relative y, m")
    axes[2].axis("equal")
    axes[2].grid(True, alpha=0.25)
    axes[2].legend(fontsize=8)

    fig.tight_layout()
    plt.show()


## 11. BLE Signal Processing and Weighting

**RSSI filtering.** At each step, we collect BLE readings from a ±0.75 s window. For every beacon in that window, we keep the strongest RSSI value. This reduces the effect of short signal dropouts.

**Beacon weighting.** We use two BLE beacons and adjust their influence according to signal strength. Since RSSI is measured in dB, it is first converted to a linear value. Each beacon then receives a fraction of the total signal power:

$$p_i = 10^{\,r_i / c}, \qquad w_i = \frac{p_i}{\sum_{j} p_j}$$

We use this weighting in two parts of the pipeline. In the step-level `ble_context`, $c=20$, and the weights combine the beacon coordinates into an approximate BLE position. In the particle-filter function `update_with_ble`, $c=10$, and the weight controls how strongly each beacon likelihood affects the particles.

As a result, a stronger and less negative RSSI has more influence, but the weaker beacon is not immediately discarded. This is useful because BLE values fluctuate even when the user is standing at the same position.

*(RSSI, beacon-weight, and relative-contribution plots can be added here after running the notebook with the complete dataset.)*

The signal variation observed in our data is consistent with the BLE positioning limitations discussed by Faragher and Harle [3].


### 11.1 BLE RSSI Visualizations

The plots below show all accepted RSSI observations for every beacon and run. A one-second trailing median is added to show the local trend while keeping the raw measurements visible. These plots also show the amount of signal variation that the particle-filter measurement model must handle.


In [ ]:
# BLE RSSI observations for all runs, with a trailing median per beacon.
import matplotlib.pyplot as plt
import pandas as pd

for run_id in RUN_IDS:
    ble = GLOBAL_BLE_DATA[run_id].copy()
    ble["time_sec"] = pd.to_numeric(ble["time_sec"], errors="coerce")
    ble["rssi"] = pd.to_numeric(ble["rssi"], errors="coerce")
    ble = ble.dropna(subset=["time_sec", "rssi", "beacon_id"]).sort_values("time_sec")

    fig, ax = plt.subplots(figsize=(13, 4.5))
    for beacon_id, group in ble.groupby("beacon_id", sort=True):
        group = group.sort_values("time_sec").copy()
        ax.scatter(group["time_sec"], group["rssi"], s=10, alpha=0.28, label=f"{beacon_id} raw")

        # A time-indexed trailing median is causal and is shown only as a diagnostic trend.
        trend = group.set_index(pd.to_timedelta(group["time_sec"], unit="s"))["rssi"].rolling("1s", min_periods=1).median()
        ax.plot(group["time_sec"], trend.to_numpy(), linewidth=1.4, label=f"{beacon_id} 1 s median")

    ax.set_title(f"{run_id}: cleaned BLE RSSI observations")
    ax.set_xlabel("Movement-relative time, s")
    ax.set_ylabel("RSSI, dBm")
    ax.grid(True, alpha=0.25)
    ax.legend(ncol=2, fontsize=8, loc="best")
    fig.tight_layout()
    plt.show()


## 12. Building Map Integration

The local map JSON defines floor dimensions, walkable polygons and stair regions. Endpoint validity removes particle weight outside these regions. This constrains sampled positions but does not establish collision-free motion along each step. The following plots show the configured geometry over the supplied floor plans.

### 12.1 Walkable Areas and Staircase


In [ ]:
# Visualize the walkable polygons and staircase connectors configured for every floor.
from pathlib import Path
import json

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch, Polygon as PolygonPatch

map_config_path = ROOT / "data" / "metadata" / "map_config.json"
if not map_config_path.exists():
    raise FileNotFoundError(
        f"Map configuration not found: {map_config_path}. "
        "Run the notebook from the submission root directory."
    )

with map_config_path.open(encoding="utf-8") as handle:
    map_config = json.load(handle)

floor_configs = map_config.get("floors", {})
if not floor_configs:
    raise ValueError("No floor definitions were found in map_config.json.")

def resolve_floor_map(floor_id: str, floor_config: dict) -> Path:
    """Return the first existing floor-map path from configured and conventional names."""
    configured_name = (
        floor_config.get("image_file")
        or floor_config.get("image_path")
        or floor_config.get("map_file")
    )
    candidates = []
    if configured_name:
        candidates.append(ROOT / configured_name)
        candidates.append(ROOT / Path(configured_name).name)

    candidates.extend(
        [
            ROOT / f"Floormap_{floor_id}.png",
            ROOT / f"floormap_{floor_id}.png",
            ROOT / f"FloorMap_{floor_id}.png",
        ]
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate

    tried = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(f"No map image found for floor {floor_id}. Tried: {tried}")

floor_ids = sorted(floor_configs.keys(), key=lambda value: int(value) if str(value).isdigit() else str(value))
fig, axes = plt.subplots(
    1,
    len(floor_ids),
    figsize=(8.5 * len(floor_ids), 5.8),
    squeeze=False,
)
fig.suptitle("Configured Walkable Areas and Staircase Regions", fontsize=15)

walkable_face = "tab:green"
walkable_edge = "darkgreen"
stair_face = "tab:orange"
stair_edge = "darkorange"

for ax, floor_id in zip(axes[0], floor_ids):
    floor_config = floor_configs[floor_id]
    map_path = resolve_floor_map(str(floor_id), floor_config)
    floor_image = mpimg.imread(map_path)
    ax.imshow(floor_image)

    walkable_polygons = floor_config.get("walkable_polygons", [])
    for polygon_index, points in enumerate(walkable_polygons, start=1):
        points_array = np.asarray(points, dtype=float)
        ax.add_patch(
            PolygonPatch(
                points_array,
                closed=True,
                facecolor=walkable_face,
                edgecolor=walkable_edge,
                linewidth=1.5,
                alpha=0.24,
                zorder=3,
            )
        )
        centroid = points_array.mean(axis=0)
        ax.text(
            centroid[0],
            centroid[1],
            f"W{polygon_index}",
            ha="center",
            va="center",
            fontsize=8,
            fontweight="bold",
            zorder=4,
        )

    stair_points = map_config.get("stairs", {}).get(f"floor{floor_id}_polygon")
    if stair_points:
        stair_array = np.asarray(stair_points, dtype=float)
        ax.add_patch(
            PolygonPatch(
                stair_array,
                closed=True,
                facecolor=stair_face,
                edgecolor=stair_edge,
                linewidth=1.8,
                alpha=0.38,
                hatch="//",
                zorder=5,
            )
        )
        stair_centroid = stair_array.mean(axis=0)
        ax.text(
            stair_centroid[0],
            stair_centroid[1],
            "Stairs",
            ha="center",
            va="center",
            fontsize=8,
            fontweight="bold",
            zorder=6,
        )

    ax.set_title(f"Floor {floor_id}: {len(walkable_polygons)} walkable polygons")
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")
    ax.set_xlim(0, floor_image.shape[1])
    ax.set_ylim(floor_image.shape[0], 0)
    ax.grid(False)

legend_handles = [
    Patch(facecolor=walkable_face, edgecolor=walkable_edge, alpha=0.24, label="Walkable polygon"),
    Patch(facecolor=stair_face, edgecolor=stair_edge, alpha=0.38, hatch="//", label="Staircase region"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=(0, 0.05, 1, 0.94))
plt.show()


## 13. Selected Filtering Method: Particle Filter

The filter maintains 1,000 weighted position and map-heading alignment hypotheses. Relative heading and step length drive prediction; log-distance RSSI likelihoods update weights, and walkable polygons constrain endpoints. Systematic resampling runs when effective particle count falls below half the cloud. The known Start initializes position and floor. Stair transitions are inferred from full-recording IMU/BLE features and transferred using map stair geometry. This is an offline reconstruction; endpoint validity does not prove the swept path avoids walls. The implementation uses anisotropic pixel conversion (12.88 px/m horizontally and 18.01 px/m vertically), rather than map JSON scalar scale fields. See [architecture](docs/ARCHITECTURE.md).


In [ ]:
from __future__ import annotations
#!/usr/bin/env python3
"""Simple particle filter: PDR prediction + BLE correction + map constraints."""

from numpy import strings
from numpy import strings
from pathlib import Path
import math
import os

import numpy as np
import pandas as pd
import json

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(Path.cwd() / ".cache"))

from matplotlib.path import Path as MplPath


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

ROOT = Path.cwd()
RUN_IDS = ("R1", "R2", "R3", "R4")
RUN_ID = RUN_IDS[0]

BEACON_FILE = ROOT / "data" / "metadata" / "beacon_locations.csv"
REFERENCE_FILE = ROOT / "data" / "metadata" / "reference_points.csv"
STEPS_FILE = ROOT / "data" / "processed" / RUN_ID / "steps.csv"
BLE_FILE = ROOT / "data" / "processed" / RUN_ID / "ble_clean.csv"
IMU_FILE = ROOT / "data" / "processed" / RUN_ID / "imu_clean.csv"
OUTPUT_FILE = ROOT / "data" / "processed" / RUN_ID / "particle_filter_positions.csv"
TRANSITION_FILE = ROOT / "data" / "processed" / RUN_ID / "floor_transition_events.csv"
ERROR_FILE = ROOT / "data" / "processed" / RUN_ID / "particle_filter_errors.csv"
ERROR_SUMMARY_FILE = ROOT / "data" / "processed" / "particle_filter_error_summary.csv"

N_PARTICLES = 1000
BLE_WINDOW_SEC = 1.0
# Log-distance path-loss model parameters used to turn RSSI into an expected
# distance (and vice versa): reference transmit power, path-loss exponent,
# and the assumed standard deviation of RSSI measurement noise (dB).
RSSI_STD_DB = 8.0
PATH_LOSS_N = 2.0
DEFAULT_TX_POWER = -59.0
BLE_WEIGHT = 1.0
BLE_BEACONS_USED = 2

# Floor-transition detector. These are shared method settings; the actual
# pause threshold, BLE margin, and IMU feature thresholds are estimated from
# each recording. No reference checkpoint times or coordinates are used.
TRANSITION_FEATURE_GRID_SEC = 0.25
TRANSITION_FEATURE_WINDOW_SEC = 2.0
TRANSITION_PRE_BLE_SEC = 4.0
TRANSITION_POST_DELAY_SEC = 3.0
TRANSITION_POST_BLE_SEC = 8.0
TRANSITION_MIN_DURATION_SEC = 6.0
TRANSITION_MIN_STEPS = 8
TRANSITION_STAIR_JITTER_M = 0.08
TRANSITION_MIN_STAIR_PARTICLES = 50

STEP_NOISE_M = 0.2
HEADING_NOISE_DEG = 7.0
HEADING_BIAS_DRIFT_DEG = 0.20
TURN_HEADING_BIAS_DRIFT_DEG = 0.45
STABLE_HEADING_NOISE_DEG = 3.0
TURN_HEADING_NOISE_DEG = 5.0
STABILIZING_HEADING_NOISE_DEG = 6.0
NEFF_RESAMPLE_RATIO = 0.50
INITIAL_HEADING_RAY_LENGTH_M = 10.0
INITIAL_HEADING_RAY_STEP_M = 0.10
INITIAL_HEADING_GRID_DEG = 5.0
INITIAL_HEADING_MODE_SPREAD_DEG = 10.0
INITIAL_HEADING_MAX_MODES = 4
USE_START_POSITION = True

# Pixel map bounds that represent the 10 m × 8 m area.
MAP_LEFT_PX = 0
MAP_RIGHT_PX = 1215
MAP_TOP_PX = 0
MAP_BOTTOM_PX = 490

PIXELS_PER_METER_X = 12.88
PIXELS_PER_METER_Y = 18.01

ROOM_WIDTH_M = (MAP_RIGHT_PX - MAP_LEFT_PX) / PIXELS_PER_METER_X
ROOM_HEIGHT_M = (MAP_BOTTOM_PX - MAP_TOP_PX) / PIXELS_PER_METER_Y

MAP_CONFIG_FILE = ROOT / "data" / "metadata" / "map_config.json"
FLOOR_ID = "0"


def set_run(run_id):
    """Switch all run-scoped file paths and the active floor ID to the given run."""
    global RUN_ID, STEPS_FILE, BLE_FILE, IMU_FILE, OUTPUT_FILE, TRANSITION_FILE, ERROR_FILE, FLOOR_ID

    RUN_ID = run_id
    STEPS_FILE = ROOT / "data" / "processed" / RUN_ID / "steps.csv"
    BLE_FILE = ROOT / "data" / "processed" / RUN_ID / "ble_clean.csv"
    IMU_FILE = ROOT / "data" / "processed" / RUN_ID / "imu_clean.csv"
    OUTPUT_FILE = ROOT / "data" / "processed" / RUN_ID / "particle_filter_positions.csv"
    TRANSITION_FILE = ROOT / "data" / "processed" / RUN_ID / "floor_transition_events.csv"
    ERROR_FILE = ROOT / "data" / "processed" / RUN_ID / "particle_filter_errors.csv"

    refs = pd.read_csv(REFERENCE_FILE)
    refs = refs[refs["run_id"] == RUN_ID]
    if not refs.empty:
        FLOOR_ID = str(refs.iloc[0]["floor"])


# ---------------------------------------------------------------------
# Coordinate conversion
# ---------------------------------------------------------------------

def pixel_to_meter(x_px, y_px):
    """Convert map pixels to metres with the origin at the bottom-left."""
    x_m = (x_px - MAP_LEFT_PX) / PIXELS_PER_METER_X
    y_m = (MAP_BOTTOM_PX - y_px) / PIXELS_PER_METER_Y
    return x_m, y_m


def meter_to_pixel(x_m, y_m):
    """Convert metres back to map pixels."""
    x_px = MAP_LEFT_PX + x_m * PIXELS_PER_METER_X
    y_px = MAP_BOTTOM_PX - y_m * PIXELS_PER_METER_Y
    return x_px, y_px


# ---------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------

def load_beacon_floor_map(csv_path=BEACON_FILE):
    """Reads the beacon CSV and returns a dictionary mapping beacon_id to floor."""
    try:
        beacons_df = pd.read_csv(csv_path)
        return dict(zip(beacons_df["beacon_id"], beacons_df["floor"].astype(str)))
    except Exception as e:
        print(f"Warning: Could not load beacon map. Error: {e}")
        return {}


BEACON_FLOORS = load_beacon_floor_map()


def load_steps():
    """Load a run's detected steps CSV, dropping rows with missing time/length/heading."""
    steps = pd.read_csv(STEPS_FILE)
    return steps.dropna(subset=["time_sec", "step_length_m", "heading_deg"])


def load_ble():
    """Load a run's cleaned BLE CSV, de-duplicated on time/beacon/address/rssi."""
    ble = pd.read_csv(BLE_FILE)
    return ble.drop_duplicates(
        subset=["time_sec", "beacon_id", "address", "rssi"]
    )


def load_imu():
    """Load the cleaned IMU stream used only for data-driven floor transitions."""
    if not IMU_FILE.exists():
        return pd.DataFrame(columns=["time_sec", "sensor", "x", "y", "z"])
    imu = pd.read_csv(IMU_FILE)
    required = ["time_sec", "sensor", "x", "y", "z"]
    return imu.dropna(subset=[column for column in required if column in imu.columns])


def load_beacons(floor_id=None):
    """Read beacon pixel positions and convert them to metres.
    If floor_id is given, only return beacons on that floor."""
    beacons = pd.read_csv(BEACON_FILE).dropna(subset=["x", "y"])
    result = {}

    for row in beacons.itertuples():
        # Skip beacons that are not on the requested floor
        if floor_id is not None:
            beacon_floor = str(getattr(row, "floor", ""))
            if beacon_floor != str(floor_id):
                continue

        x_m, y_m = pixel_to_meter(float(row.x), float(row.y))
        tx_power = float(getattr(row, "tx_power", DEFAULT_TX_POWER))
        result[str(row.beacon_id)] = (x_m, y_m, tx_power)

    return result



def load_start_position():
    """Use the route's Start reference point, or fall back to map centre."""
    if REFERENCE_FILE.exists():
        refs = pd.read_csv(REFERENCE_FILE)
        match = refs[
            (refs["run_id"] == RUN_ID) &
            (refs["ref_id"].astype(str).str.lower() == "start")
        ]
        if not match.empty:
            row = match.iloc[0]
            return pixel_to_meter(float(row["x"]), float(row["y"]))

    return ROOM_WIDTH_M / 2, ROOM_HEIGHT_M / 2


def load_imu_start_offset():
    """Return the first cleaned IMU timestamp for movement-time alignment."""
    imu_file = ROOT / "data" / "processed" / RUN_ID / "imu_clean.csv"
    if not imu_file.exists():
        return 0.0

    imu = pd.read_csv(imu_file)
    if imu.empty or "time_sec" not in imu:
        return 0.0

    return float(imu["time_sec"].min())


# ---------------------------------------------------------------------
# Particle filter operations
# ---------------------------------------------------------------------

def initial_map_heading_candidates(start_x, start_y, map_data):
    """Find plausible initial map directions without using the reference route."""
    angles = np.arange(0.0, 360.0, INITIAL_HEADING_GRID_DEG)
    distances = np.arange(
        INITIAL_HEADING_RAY_STEP_M,
        INITIAL_HEADING_RAY_LENGTH_M + INITIAL_HEADING_RAY_STEP_M,
        INITIAL_HEADING_RAY_STEP_M,
    )
    scores = np.zeros(len(angles), dtype=float)

    for index, angle_deg in enumerate(angles):
        angle = math.radians(float(angle_deg))
        probe = np.zeros((len(distances), 3), dtype=float)
        probe[:, 0] = start_x + distances * math.sin(angle)
        probe[:, 1] = start_y + distances * math.cos(angle)
        valid = valid_particle_mask(probe, map_data)
        first_invalid = np.flatnonzero(~valid)
        valid_count = int(first_invalid[0]) if len(first_invalid) else len(valid)
        scores[index] = valid_count * INITIAL_HEADING_RAY_STEP_M

    smooth_scores = (scores + np.roll(scores, 1) + np.roll(scores, -1)) / 3.0
    maximum = float(smooth_scores.max())
    active = smooth_scores >= max(1.0, 0.82 * maximum)

    # Group active directions on the circular angle grid.
    groups = []
    visited = np.zeros(len(active), dtype=bool)
    for seed in np.flatnonzero(active):
        if visited[seed]:
            continue
        group = []
        stack = [int(seed)]
        while stack:
            item = stack.pop()
            if visited[item] or not active[item]:
                continue
            visited[item] = True
            group.append(item)
            stack.extend(((item - 1) % len(active), (item + 1) % len(active)))
        groups.append(group)

    candidates = []
    for group in groups:
        group_angles = np.deg2rad(angles[group])
        group_weights = smooth_scores[group]
        x = float(np.sum(group_weights * np.cos(group_angles)))
        y = float(np.sum(group_weights * np.sin(group_angles)))
        angle = math.degrees(math.atan2(y, x)) % 360.0
        score = float(np.sum(group_weights - 0.82 * maximum + 0.1))
        candidates.append((angle, score))

    candidates.sort(key=lambda item: item[1], reverse=True)
    candidates = candidates[:INITIAL_HEADING_MAX_MODES]
    if not candidates:
        candidates = [(angle, 1.0) for angle in (0.0, 90.0, 180.0, 270.0)]
    return candidates

def initialize_particles(start_x, start_y, map_data, rng):
    """Initialize position and map-heading hypotheses around a known start."""
    particles = np.zeros((N_PARTICLES, 3))
    particles[:, 0] = rng.normal(start_x, 0.25, N_PARTICLES)
    particles[:, 1] = rng.normal(start_y, 0.25, N_PARTICLES)

    candidates = initial_map_heading_candidates(start_x, start_y, map_data)
    candidate_angles = np.asarray([item[0] for item in candidates], dtype=float)
    candidate_scores = np.asarray([item[1] for item in candidates], dtype=float)
    # Map geometry may be ambiguous at intersections. Keep each plausible mode
    # alive initially instead of treating a wider polygon opening as the route.
    probabilities = np.full(len(candidate_angles), 1.0 / len(candidate_angles))
    chosen = rng.choice(len(candidate_angles), size=N_PARTICLES, p=probabilities)
    particles[:, 2] = np.deg2rad(
        candidate_angles[chosen]
        + rng.normal(0.0, INITIAL_HEADING_MODE_SPREAD_DEG, N_PARTICLES)
    )

    weights = np.full(N_PARTICLES, 1.0 / N_PARTICLES)
    return particles, weights, candidates

def initialize_particles_from_map(map_data, rng):
    """Initialize particles by uniform rejection sampling inside the floor's walkable polygons."""
    particles = np.zeros((N_PARTICLES, 3))
    count = 0

    while count < N_PARTICLES:
        sample = np.zeros((N_PARTICLES, 3))
        x_px = rng.uniform(0, map_data["width_px"], N_PARTICLES)
        y_px = rng.uniform(0, map_data["height_px"], N_PARTICLES)
        sample[:, 0], sample[:, 1] = pixel_to_meter(x_px, y_px)

        inside = valid_particle_mask(sample, map_data)
        valid = sample[inside]
        take = min(len(valid), N_PARTICLES - count)

        particles[count:count + take] = valid[:take]
        count += take

    particles[:, 2] = rng.uniform(-math.pi, math.pi, N_PARTICLES)
    weights = np.full(N_PARTICLES, 1.0 / N_PARTICLES)
    return particles, weights


def predict(particles, step_length, relative_heading_deg, heading_mode, rng):
    """Move particles using persistent relative heading and map-alignment hypotheses."""
    if heading_mode == "turning":
        heading_noise_deg = TURN_HEADING_NOISE_DEG
        drift_deg = TURN_HEADING_BIAS_DRIFT_DEG
    elif heading_mode == "stabilizing":
        heading_noise_deg = STABILIZING_HEADING_NOISE_DEG
        drift_deg = HEADING_BIAS_DRIFT_DEG
    else:
        heading_noise_deg = STABLE_HEADING_NOISE_DEG
        drift_deg = HEADING_BIAS_DRIFT_DEG

    particles[:, 2] += rng.normal(0.0, math.radians(drift_deg), N_PARTICLES)
    heading = (
        math.radians(relative_heading_deg)
        + particles[:, 2]
        + rng.normal(0.0, math.radians(heading_noise_deg), N_PARTICLES)
    )
    distance = np.maximum(0.0, step_length + rng.normal(0.0, STEP_NOISE_M, N_PARTICLES))

    # Compass convention: 0° = map north and 90° = map east.
    particles[:, 0] += distance * np.sin(heading)
    particles[:, 1] += distance * np.cos(heading)

    return heading_noise_deg, drift_deg

def load_map_constraints(floor_id):
    """Loads walkable areas AND stair boundaries for the given floor."""
    with MAP_CONFIG_FILE.open() as file:
        config = json.load(file)

    floor = config["floors"][str(floor_id)]

    polygons = [
        MplPath(np.asarray(points, dtype=float))
        for points in floor["walkable_polygons"]
    ]

    stair_polygon = None
    stairs_config = config.get("stairs", {})
    stair_key = f"floor{floor_id}_polygon"
    if stair_key in stairs_config:
        stair_polygon = MplPath(np.asarray(stairs_config[stair_key], dtype=float))

    return {
        "polygons": polygons,
        "stair_polygon": stair_polygon,
        "width_px": float(floor["image_width_px"]),
        "height_px": float(floor["image_height_px"]),
    }


def particles_to_pixels(particles, map_data):
    """Convert particle (x, y) positions from metres to pixel coordinates."""
    x_px, y_px = meter_to_pixel(particles[:, 0], particles[:, 1])
    return np.column_stack((x_px, y_px))


def valid_particle_mask(particles, map_data):
    """Return True for particles inside any walkable polygon OR stairs."""
    particle_pixels = particles_to_pixels(particles, map_data)
    inside = np.zeros(len(particles), dtype=bool)

    # Check normal walkable areas
    for polygon in map_data["polygons"]:
        inside |= polygon.contains_points(particle_pixels, radius=1e-9)

    # Check stair areas
    if map_data.get("stair_polygon"):
        inside |= map_data["stair_polygon"].contains_points(particle_pixels, radius=1e-9)

    return inside


def remove_and_resample_invalid(particles, weights, map_data, rng):
    """Remove particles outside the walkable area and resample valid ones."""
    inside = valid_particle_mask(particles, map_data)

    valid_particles = particles[inside]
    valid_weights = weights[inside]

    if len(valid_particles) == 0:
        return False

    total = valid_weights.sum()

    if not np.isfinite(total) or total <= 0:
        valid_weights = np.full(len(valid_particles), 1.0 / len(valid_particles))
    else:
        valid_weights = valid_weights / total

    selected = rng.choice(
        len(valid_particles),
        size=len(particles),
        replace=True,
        p=valid_weights,
    )

    particles[:] = valid_particles[selected]
    weights.fill(1.0 / len(weights))
    return True



def apply_map_constraint(particles, weights, map_data):
    """Set impossible particle weights to zero without resampling immediately."""
    inside = valid_particle_mask(particles, map_data)
    weights[~inside] = 0.0
    return int(inside.sum())

def robust_mad(values):
    """Median absolute deviation, ignoring non-finite values."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return float("nan")
    median = np.median(values)
    return float(np.median(np.abs(values - median)))


def normalize_beacon_id(value):
    """Normalize numeric-looking and string beacon IDs to the metadata format."""
    raw = str(value).strip()
    try:
        numeric = float(raw)
        if numeric.is_integer():
            return str(int(numeric))
    except ValueError:
        pass
    return raw


def movement_bouts(steps):
    """Split movement events into pause-bounded bouts using an adaptive gap."""
    times = np.asarray(steps["time_sec"], dtype=float)
    if len(times) == 0:
        return [], 2.0
    if len(times) == 1:
        return [{"start_time": times[0], "end_time": times[0], "steps": 1,
                 "pre_gap": float("inf"), "post_gap": float("inf")}], 2.0

    intervals = np.diff(times)
    median_interval = float(np.median(intervals))
    interval_mad = robust_mad(intervals)
    if not np.isfinite(interval_mad):
        interval_mad = 0.0

    # Normal steps are roughly 0.4-0.7 s apart. A true pause must be much
    # longer than both the current cadence and its natural variability.
    pause_gap = max(
        2.0,
        3.5 * median_interval,
        median_interval + 8.0 * interval_mad,
    )

    starts = [0]
    ends = []
    split_indexes = np.flatnonzero(intervals > pause_gap)
    for index in split_indexes:
        ends.append(int(index))
        starts.append(int(index + 1))
    ends.append(len(times) - 1)

    bouts = []
    for bout_index, (start_index, end_index) in enumerate(zip(starts, ends)):
        pre_gap = float("inf") if bout_index == 0 else float(times[start_index] - times[start_index - 1])
        post_gap = float("inf") if bout_index == len(starts) - 1 else float(times[end_index + 1] - times[end_index])
        bouts.append({
            "start_time": float(times[start_index]),
            "end_time": float(times[end_index]),
            "steps": int(end_index - start_index + 1),
            "pre_gap": pre_gap,
            "post_gap": post_gap,
        })
    return bouts, pause_gap


def compute_transition_features(imu, steps):
    """Create gravity-aligned vertical, gyro, and tilt features on a causal grid."""
    if imu.empty:
        return pd.DataFrame()

    accel = imu[imu["sensor"] == "accel"][["time_sec", "x", "y", "z"]].copy()
    gyro = imu[imu["sensor"] == "gyro"][["time_sec", "x", "y", "z"]].copy()
    orientation = imu[imu["sensor"] == "imu_processed"][["time_sec", "x", "y", "z"]].copy()

    if accel.empty or gyro.empty or orientation.empty:
        return pd.DataFrame()

    accel = accel.sort_values("time_sec").drop_duplicates("time_sec")
    gyro = gyro.sort_values("time_sec").drop_duplicates("time_sec")
    orientation = (
        orientation.groupby("time_sec", as_index=False)[["x", "y", "z"]]
        .median()
        .sort_values("time_sec")
    )

    accel_values = accel[["x", "y", "z"]].to_numpy(dtype=float)
    accel_times = accel["time_sec"].to_numpy(dtype=float)
    sample_dt = float(np.median(np.diff(accel_times))) if len(accel_times) > 1 else 0.01
    sample_dt = max(sample_dt, 1e-4)

    # A causal low-pass estimate of gravity makes vertical acceleration
    # independent of whether the phone is held flat, tilted, or in a pocket.
    gravity_tau_sec = 1.2
    alpha = 1.0 - math.exp(-sample_dt / gravity_tau_sec)
    gravity = np.empty_like(accel_values)
    gravity[0] = accel_values[0]
    for index in range(1, len(accel_values)):
        gravity[index] = (1.0 - alpha) * gravity[index - 1] + alpha * accel_values[index]

    gravity_norm = np.linalg.norm(gravity, axis=1, keepdims=True)
    gravity_unit = gravity / np.maximum(gravity_norm, 1e-6)
    dynamic_accel = accel_values - gravity
    vertical_dynamic = np.sum(dynamic_accel * gravity_unit, axis=1)

    gyro_values = gyro[["x", "y", "z"]].to_numpy(dtype=float)
    gyro_norm = np.linalg.norm(gyro_values, axis=1)

    max_time = max(
        float(accel["time_sec"].max()),
        float(gyro["time_sec"].max()),
        float(orientation["time_sec"].max()),
        float(steps["time_sec"].max()) if not steps.empty else 0.0,
    )
    grid = np.arange(0.0, max_time + TRANSITION_FEATURE_GRID_SEC / 2, TRANSITION_FEATURE_GRID_SEC)
    features = pd.DataFrame({"time_sec": grid})
    features["vertical_dynamic"] = np.interp(grid, accel_times, vertical_dynamic)
    features["gyro_norm"] = np.interp(grid, gyro["time_sec"], gyro_norm)
    features["pitch_deg"] = np.interp(grid, orientation["time_sec"], orientation["y"])
    features["roll_deg"] = np.interp(grid, orientation["time_sec"], orientation["z"])

    window = max(3, int(round(TRANSITION_FEATURE_WINDOW_SEC / TRANSITION_FEATURE_GRID_SEC)))
    min_periods = max(2, window // 2)
    features["vertical_rms"] = features["vertical_dynamic"].rolling(
        window, min_periods=min_periods
    ).apply(lambda values: math.sqrt(float(np.mean(np.square(values)))), raw=True)
    features["gyro_rms"] = features["gyro_norm"].rolling(
        window, min_periods=min_periods
    ).apply(lambda values: math.sqrt(float(np.mean(np.square(values)))), raw=True)
    features["pitch_std"] = features["pitch_deg"].rolling(window, min_periods=min_periods).std()
    features["roll_std"] = features["roll_deg"].rolling(window, min_periods=min_periods).std()
    return features


def summarize_bout_features(features, bout):
    """Summarize a movement bout's IMU features (vertical/gyro RMS quantiles, tilt variability) for floor-transition detection."""
    if features.empty:
        return {
            "vertical_q75": float("nan"),
            "gyro_q75": float("nan"),
            "tilt_median": float("nan"),
        }
    window = features[
        (features["time_sec"] >= bout["start_time"]) &
        (features["time_sec"] <= bout["end_time"])
    ]
    if window.empty:
        return {
            "vertical_q75": float("nan"),
            "gyro_q75": float("nan"),
            "tilt_median": float("nan"),
        }
    return {
        "vertical_q75": float(window["vertical_rms"].quantile(0.75)),
        "gyro_q75": float(window["gyro_rms"].quantile(0.75)),
        "tilt_median": float((window["pitch_std"] + window["roll_std"]).median()),
    }


def estimate_ble_margin_threshold(ble):
    """Estimate a run-specific RSSI margin from short-term beacon fluctuations."""
    absolute_changes = []
    for _, group in ble.sort_values("time_sec").groupby("beacon_id"):
        changes = np.abs(np.diff(group["rssi"].to_numpy(dtype=float)))
        absolute_changes.extend(changes[np.isfinite(changes)])
    if not absolute_changes:
        return 6.0
    # The 75th percentile is conservative enough to reject ordinary RSSI
    # fluctuations while remaining data-adaptive for each phone/run.
    return max(4.0, float(np.quantile(absolute_changes, 0.75)))


def floor_ble_scores(ble, start_time, end_time):
    """Robust floor evidence using the two strongest per-beacon medians."""
    window = ble[
        (ble["time_sec"] >= start_time) &
        (ble["time_sec"] <= end_time)
    ].copy()
    if window.empty:
        return {}

    window["beacon_id"] = window["beacon_id"].map(normalize_beacon_id)
    window["floor"] = window["beacon_id"].map(BEACON_FLOORS)
    window = window.dropna(subset=["floor", "rssi"])

    scores = {}
    for floor_id, group in window.groupby("floor"):
        medians = group.groupby("beacon_id")["rssi"].median().sort_values(ascending=False)
        strongest = medians.head(2)
        scores[str(floor_id)] = {
            "score_dbm": float(strongest.mean()),
            "beacons": int(len(medians)),
            "samples": int(len(group)),
        }
    return scores


def detect_floor_transitions(steps, imu, ble, initial_floor_id):
    """Detect pause-bounded stair transitions from IMU and BLE only.

    Reference points are deliberately not read here. The map is used later to
    move particles between stair polygons, but transition timing is inferred
    exclusively from recorded IMU, step timing, and BLE floor evidence.
    """
    bouts, pause_gap = movement_bouts(steps)
    if len(bouts) < 3:
        return [], pd.DataFrame()

    features = compute_transition_features(imu, steps)
    for bout in bouts:
        bout.update(summarize_bout_features(features, bout))
        bout["duration_sec"] = bout["end_time"] - bout["start_time"]

    feature_frame = pd.DataFrame(bouts)
    feature_medians = {
        "vertical_q75": float(feature_frame["vertical_q75"].median()),
        "gyro_q75": float(feature_frame["gyro_q75"].median()),
        "tilt_median": float(feature_frame["tilt_median"].median()),
        "duration_sec": float(feature_frame["duration_sec"].median()),
        "steps": float(feature_frame["steps"].median()),
    }
    ble_margin_threshold = estimate_ble_margin_threshold(ble)
    current_floor = str(initial_floor_id)
    transitions = []
    diagnostics = []
    all_floors = sorted(set(str(value) for value in BEACON_FLOORS.values()))
    recording_end = float(max(ble["time_sec"].max(), steps["time_sec"].max()))

    for index, bout in enumerate(bouts):
        # A staircase traversal should be bounded by genuine pauses. This also
        # prevents the first and last ordinary route segments being candidates.
        bounded_by_pauses = (
            index > 0 and index < len(bouts) - 1 and
            bout["pre_gap"] >= pause_gap and bout["post_gap"] >= pause_gap
        )

        pre_scores = floor_ble_scores(
            ble,
            max(0.0, bout["start_time"] - TRANSITION_PRE_BLE_SEC),
            bout["start_time"],
        )
        post_scores = floor_ble_scores(
            ble,
            bout["end_time"] + TRANSITION_POST_DELAY_SEC,
            min(recording_end, bout["end_time"] + TRANSITION_POST_DELAY_SEC + TRANSITION_POST_BLE_SEC),
        )

        other_floors = [
            floor for floor in all_floors
            if floor != current_floor and floor in post_scores
        ]
        destination_floor = None
        post_margin = float("nan")
        pre_current_margin = float("nan")
        margin_change = float("nan")
        destination_beacons = 0
        destination_samples = 0

        if current_floor in post_scores and other_floors:
            destination_floor = max(
                other_floors,
                key=lambda floor: post_scores[floor]["score_dbm"],
            )
            post_margin = (
                post_scores[destination_floor]["score_dbm"] -
                post_scores[current_floor]["score_dbm"]
            )
            destination_beacons = post_scores[destination_floor]["beacons"]
            destination_samples = post_scores[destination_floor]["samples"]
            if current_floor in pre_scores and destination_floor in pre_scores:
                pre_current_margin = (
                    pre_scores[current_floor]["score_dbm"] -
                    pre_scores[destination_floor]["score_dbm"]
                )
                margin_change = post_margin + pre_current_margin
            else:
                margin_change = post_margin

        imu_votes = sum([
            np.isfinite(bout["vertical_q75"]) and bout["vertical_q75"] >= feature_medians["vertical_q75"],
            np.isfinite(bout["gyro_q75"]) and bout["gyro_q75"] >= feature_medians["gyro_q75"],
            np.isfinite(bout["tilt_median"]) and bout["tilt_median"] >= feature_medians["tilt_median"],
        ])
        duration_ok = bout["duration_sec"] >= max(
            TRANSITION_MIN_DURATION_SEC,
            0.75 * feature_medians["duration_sec"],
        )
        steps_ok = bout["steps"] >= max(
            TRANSITION_MIN_STEPS,
            0.75 * feature_medians["steps"],
        )
        ble_ok = (
            destination_floor is not None and
            np.isfinite(post_margin) and
            np.isfinite(margin_change) and
            post_margin >= ble_margin_threshold and
            margin_change >= 1.5 * ble_margin_threshold and
            destination_samples >= 5 and
            (destination_beacons >= 2 or post_margin >= 1.5 * ble_margin_threshold)
        )
        accepted = bool(
            bounded_by_pauses and duration_ok and steps_ok and imu_votes >= 2 and ble_ok
        )

        diagnostic = {
            "run_id": RUN_ID,
            "bout_index": index,
            "start_time_sec": bout["start_time"],
            "end_time_sec": bout["end_time"],
            "duration_sec": bout["duration_sec"],
            "step_count": bout["steps"],
            "pause_gap_threshold_sec": pause_gap,
            "pre_gap_sec": bout["pre_gap"],
            "post_gap_sec": bout["post_gap"],
            "from_floor": current_floor,
            "candidate_to_floor": destination_floor or "",
            "vertical_q75": bout["vertical_q75"],
            "gyro_q75": bout["gyro_q75"],
            "tilt_median": bout["tilt_median"],
            "imu_votes": imu_votes,
            "ble_margin_threshold_db": ble_margin_threshold,
            "post_destination_margin_db": post_margin,
            "pre_current_margin_db": pre_current_margin,
            "ble_margin_change_db": margin_change,
            "destination_beacons": destination_beacons,
            "destination_samples": destination_samples,
            "accepted_transition": accepted,
        }
        diagnostics.append(diagnostic)

        if accepted:
            confidence = min(
                1.0,
                0.35 * (imu_votes / 3.0) +
                0.35 * min(post_margin / max(ble_margin_threshold, 1e-6), 2.0) / 2.0 +
                0.30 * min(margin_change / max(1.5 * ble_margin_threshold, 1e-6), 2.0) / 2.0,
            )
            transitions.append({
                "run_id": RUN_ID,
                "start_time_sec": bout["start_time"],
                "end_time_sec": bout["end_time"],
                "from_floor": current_floor,
                "to_floor": destination_floor,
                "confidence": confidence,
                "post_destination_margin_db": post_margin,
                "ble_margin_change_db": margin_change,
                "imu_votes": imu_votes,
            })
            current_floor = destination_floor

    return transitions, pd.DataFrame(diagnostics)


def stair_vertices(floor_id):
    """Load the pixel-coordinate polygon of the staircase region for the given floor from the map config."""
    with MAP_CONFIG_FILE.open() as file:
        config = json.load(file)
    key = f"floor{floor_id}_polygon"
    points = config.get("stairs", {}).get(key)
    if points is None:
        raise KeyError(f"No staircase polygon configured for floor {floor_id}")
    return np.asarray(points, dtype=float)


def sample_uniform_in_stair(floor_id, count, rng):
    """Sample metre coordinates inside the configured rectangular stair polygon."""
    vertices = stair_vertices(floor_id)
    x_px = rng.uniform(vertices[:, 0].min(), vertices[:, 0].max(), count)
    y_px = rng.uniform(vertices[:, 1].min(), vertices[:, 1].max(), count)
    x_m, y_m = pixel_to_meter(x_px, y_px)
    return np.column_stack((x_m, y_m))


def condition_particles_on_stair(particles, weights, floor_id, map_data, rng):
    """Condition the current-floor cloud on the mapped staircase region."""
    pixel_points = particles_to_pixels(particles, map_data)
    mask = map_data["stair_polygon"].contains_points(pixel_points, radius=1e-9)
    stair_count = int(mask.sum())

    if stair_count >= TRANSITION_MIN_STAIR_PARTICLES:
        stair_weights = weights[mask]
        total = stair_weights.sum()
        if not np.isfinite(total) or total <= 0:
            stair_weights = np.full(stair_count, 1.0 / stair_count)
        else:
            stair_weights = stair_weights / total
        indexes = rng.choice(stair_count, size=len(particles), replace=True, p=stair_weights)
        particles[:] = particles[mask][indexes]
    else:
        # The sensor data says the user entered a staircase, but accumulated PDR
        # error may have placed the cloud outside it. Re-localize to the whole
        # staircase polygon, not to a checkpoint or exact coordinate.
        source_indexes = rng.choice(len(particles), size=len(particles), replace=True, p=weights / weights.sum())
        heading_bias = particles[source_indexes, 2].copy()
        particles[:, :2] = sample_uniform_in_stair(floor_id, len(particles), rng)
        particles[:, 2] = heading_bias

    weights.fill(1.0 / len(weights))
    return stair_count


def keep_particles_inside_stair(particles, floor_id, rng):
    """Represent stair traversal as vertical motion with only small 2D uncertainty."""
    vertices = stair_vertices(floor_id)
    x_min, x_max = vertices[:, 0].min(), vertices[:, 0].max()
    y_min, y_max = vertices[:, 1].min(), vertices[:, 1].max()
    x_px, y_px = meter_to_pixel(particles[:, 0], particles[:, 1])
    x_px += rng.normal(0.0, TRANSITION_STAIR_JITTER_M * PIXELS_PER_METER_X, len(particles))
    y_px += rng.normal(0.0, TRANSITION_STAIR_JITTER_M * PIXELS_PER_METER_Y, len(particles))
    x_px = np.clip(x_px, x_min, x_max)
    y_px = np.clip(y_px, y_min, y_max)
    particles[:, 0], particles[:, 1] = pixel_to_meter(x_px, y_px)


def stair_exit_edge(floor_id):
    """Find the staircase edge connected to ordinary walkable space."""
    vertices = stair_vertices(floor_id)
    x_min, x_max = vertices[:, 0].min(), vertices[:, 0].max()
    y_min, y_max = vertices[:, 1].min(), vertices[:, 1].max()
    map_data = load_map_constraints(floor_id)

    edge_points = {
        "left": np.column_stack((np.full(25, x_min - 3.0), np.linspace(y_min, y_max, 25))),
        "right": np.column_stack((np.full(25, x_max + 3.0), np.linspace(y_min, y_max, 25))),
        "top": np.column_stack((np.linspace(x_min, x_max, 25), np.full(25, y_min - 3.0))),
        "bottom": np.column_stack((np.linspace(x_min, x_max, 25), np.full(25, y_max + 3.0))),
    }

    scores = {}
    for name, points in edge_points.items():
        inside = np.zeros(len(points), dtype=bool)
        for polygon in map_data["polygons"]:
            inside |= polygon.contains_points(points, radius=1e-9)
        scores[name] = int(inside.sum())

    return max(scores, key=scores.get), (x_min, x_max, y_min, y_max)


def transfer_particles_between_stairs(particles, weights, from_floor, to_floor, rng):
    """Move the cloud to the map-derived exit of the destination staircase.

    The transition detector decides *when* the floor changes from IMU and BLE.
    The map only decides where the destination stair connects to its corridor.
    No reference checkpoint or exact S4 coordinate is used.
    """
    destination = stair_vertices(to_floor)
    dx_min, dx_max = destination[:, 0].min(), destination[:, 0].max()
    dy_min, dy_max = destination[:, 1].min(), destination[:, 1].max()
    edge, _ = stair_exit_edge(to_floor)

    count = len(particles)
    if edge in {"top", "bottom"}:
        x_px = rng.uniform(dx_min, dx_max, count)
        y_boundary = dy_min - 3.0 if edge == "top" else dy_max + 3.0
        y_px = rng.normal(y_boundary, 3.0, count)
    else:
        y_px = rng.uniform(dy_min, dy_max, count)
        x_boundary = dx_min - 3.0 if edge == "left" else dx_max + 3.0
        x_px = rng.normal(x_boundary, 3.0, count)

    particles[:, 0], particles[:, 1] = pixel_to_meter(x_px, y_px)
    normalize(weights)

def nearby_ble(ble, time_sec):
    """Keep the strongest RSSI for each beacon near the step time."""
    nearby = ble[
        (ble["time_sec"] >= time_sec - BLE_WINDOW_SEC / 2) &
        (ble["time_sec"] <= time_sec + BLE_WINDOW_SEC / 2)
    ]

    if nearby.empty:
        return {}

    return (
        nearby.groupby("beacon_id")["rssi"]
        .max()
        .astype(float)
        .to_dict()
    )



def update_with_ble(particles, weights, observations, beacons):
    """Compare measured RSSI with RSSI expected at each particle.
    Stronger beacons (higher RSSI) get more influence."""
    if not observations:
        return

    # --- Step 1: Figure out how much weight each beacon gets ---
    # Stronger RSSI = closer beacon = more trustworthy
    # Convert RSSI to a linear scale so we can compute proportions
    valid_beacons = [
        (bid, rssi) for bid, rssi in observations.items()
        if bid in beacons
    ]
    if not valid_beacons:
        return

    # Convert RSSI from dB to linear power (higher = stronger)
    linear_powers = [10.0 ** (rssi / 10.0) for _, rssi in valid_beacons]
    total_power = sum(linear_powers)

    # Each beacon's share: e.g. if beacon A is twice as strong as B,
    # A gets ~0.67 weight and B gets ~0.33 weight
    beacon_shares = [p / total_power for p in linear_powers]

    # --- Step 2: Apply each beacon's BLE correction ---
    for (beacon_id, measured_rssi), share in zip(valid_beacons, beacon_shares):
        bx, by, tx_power = beacons[beacon_id]
        distance = np.hypot(particles[:, 0] - bx, particles[:, 1] - by)
        distance = np.maximum(distance, 0.25)

        expected_rssi = tx_power - 10 * PATH_LOSS_N * np.log10(distance)
        error = measured_rssi - expected_rssi

                # Gaussian likelihood of the measured RSSI given this particle's position.
        ble_likelihood = np.exp(-0.5 * (error / RSSI_STD_DB) ** 2)

        # Scale the BLE_WEIGHT by this beacon's share
        # Stronger beacon → higher share → more pull
        beacon_weight = BLE_WEIGHT * share
        weights *= ble_likelihood ** beacon_weight



def normalize(weights):
    """Normalize a particle-weight array in place so that it sums to 1, resetting to uniform if degenerate."""
    total = weights.sum()
    if not np.isfinite(total) or total <= 0:
        weights.fill(1.0 / len(weights))
    else:
        weights /= total


def estimate_position(particles, weights, map_data=None):
    """Return the weighted-mean particle position, snapped to the nearest particle if that mean falls outside the walkable area."""
    x = np.average(particles[:, 0], weights=weights)
    y = np.average(particles[:, 1], weights=weights)

    if map_data is None:
        return x, y

    x_px, y_px = meter_to_pixel(x, y)
    point = (x_px, y_px)

    for polygon in map_data["polygons"]:
        if polygon.contains_point(point, radius=1e-9):
            return x, y
    if map_data.get("stair_polygon") and map_data["stair_polygon"].contains_point(point, radius=1e-9):
        return x, y

    best = np.argmax(weights)
    return particles[best, 0], particles[best, 1]


def weighted_map_heading_deg(particles, weights, relative_heading_deg):
    """Return the weighted circular mean of the current map heading."""
    angles = particles[:, 2] + math.radians(relative_heading_deg)
    x = float(np.sum(weights * np.cos(angles)))
    y = float(np.sum(weights * np.sin(angles)))
    return math.degrees(math.atan2(y, x)) % 360.0


def systematic_resample(particles, weights, rng):
    """Resample particles proportional to their weights using systematic resampling, then reset weights to uniform."""
    positions = (rng.random() + np.arange(N_PARTICLES)) / N_PARTICLES
    indexes = np.searchsorted(np.cumsum(weights), positions)
    particles[:] = particles[indexes]
    weights.fill(1.0 / N_PARTICLES)


def calculate_errors(output):
    """Compare PF estimates with manual reference points."""
    refs = pd.read_csv(REFERENCE_FILE)
    refs = refs[refs["run_id"] == RUN_ID].dropna(subset=["time_sec", "x", "y"])
    pf_time_offset = load_imu_start_offset()

    rows = []
    for ref in refs.itertuples():
        ref_time = float(ref.time_sec)
        ref_x = float(ref.x)
        ref_y = float(ref.y)

        reference_floor = str(ref.floor)
        floor_output = [row for row in output if str(row.get("floor", "")) == reference_floor]
        if not floor_output:
            continue
        pf = min(
            floor_output,
            key=lambda row: abs((row["time_sec"] - pf_time_offset) - ref_time),
        )
        pf_movement_time = pf["time_sec"] - pf_time_offset

        dx_px = pf["x_px"] - ref_x
        dy_px = pf["y_px"] - ref_y
        error_px = math.hypot(dx_px, dy_px)

        dx_m = dx_px / PIXELS_PER_METER_X
        dy_m = dy_px / PIXELS_PER_METER_Y
        error_m = math.hypot(dx_m, dy_m)

        rows.append({
            "run_id": RUN_ID,
            "ref_id": ref.ref_id,
            "ref_time_sec": ref_time,
            "ref_floor": reference_floor,
            "pf_floor": str(pf.get("floor", "")),
            "pf_time_sec": pf["time_sec"],
            "pf_movement_time_sec": pf_movement_time,
            "time_error_sec": pf_movement_time - ref_time,
            "ref_x_px": ref_x,
            "ref_y_px": ref_y,
            "pf_x_px": pf["x_px"],
            "pf_y_px": pf["y_px"],
            "error_px": error_px,
            "error_m": error_m,
        })

    errors = pd.DataFrame(rows)
    errors.to_csv(ERROR_FILE, index=False)
    return errors


def summarize_error_rows(errors):
    """Summarize a run's localization errors into reference-point count, mean, RMSE, and max error (pixels and metres)."""
    if errors.empty:
        return {k: "" for k in ["reference_points", "mean_error_px", "mean_error_m", "rmse_error_px", "rmse_error_m", "max_error_px", "max_error_m"]}
    
    return {
        "reference_points": len(errors),
        "mean_error_px": errors["error_px"].mean(),
        "mean_error_m": errors["error_m"].mean(),
        "rmse_error_px": math.sqrt((errors["error_px"] ** 2).mean()),
        "rmse_error_m": math.sqrt((errors["error_m"] ** 2).mean()),
    }


def calculate_overall_errors(run_ids):
    """Compute per-run and pooled overall localization error summaries across all runs."""
    summary_rows = []
    all_errors = []

    for run_id in run_ids:
        errors = GLOBAL_PF_ERRORS.get(run_id, pd.DataFrame())
        if errors.empty:
            continue

            pass
        if not errors.empty:
            all_errors.append(errors)

        summary_rows.append({"run_id": run_id, **summarize_error_rows(errors)})

    combined = pd.concat(all_errors, ignore_index=True) if all_errors else pd.DataFrame()
    summary_rows.append({"run_id": "OVERALL", **summarize_error_rows(combined)})

    summary = pd.DataFrame(summary_rows)
    pass
    return summary


# ---------------------------------------------------------------------
# Main filter
# ---------------------------------------------------------------------

def run_autonomous_particle_filter(
    steps,
    ble,
    beacons,
    initial_floor_id,
    start_position,
    transitions,
    rng,
):
    """Run the teammate PF unchanged outside sensor-detected stair intervals."""
    floor_id = str(initial_floor_id)
    map_data = load_map_constraints(floor_id)
    particles, weights, initial_heading_candidates = initialize_particles(
        *start_position, map_data, rng
    )
    output = []
    particle_states = []

    transition_index = 0
    active_transition = None
    transition_source_particle_count = 0

    for step in steps.itertuples():
        step_time = float(step.time_sec)

        # Transition timing comes only from IMU + BLE. The map is used after
        # detection to condition and transfer the cloud through mapped stairs.
        if (
            active_transition is None
            and transition_index < len(transitions)
            and step_time >= transitions[transition_index]["start_time_sec"]
        ):
            candidate = transitions[transition_index]
            if str(candidate["from_floor"]) == floor_id:
                active_transition = candidate
                transition_source_particle_count = condition_particles_on_stair(
                    particles, weights, floor_id, map_data, rng
                )
                print(
                    f"\n>>> [{step_time:.1f}s] Stair transition detected from data: "
                    f"Floor {candidate['from_floor']} -> Floor {candidate['to_floor']} "
                    f"(confidence={candidate['confidence']:.2f}) <<<\n"
                )
            else:
                transition_index += 1

        transition_state = "stable"
        observations = {}
        valid_particles_after_prediction = int(valid_particle_mask(particles, map_data).sum())
        rollback_occurred = False
        heading_noise_deg = 0.0
        heading_drift_deg = 0.0

        if active_transition is not None:
            if step_time < active_transition["end_time_sec"]:
                # Stair impacts are mostly vertical. Preserve the teammate's
                # heading state but prevent ordinary 2D PDR from overshooting.
                transition_state = "stairs"
                keep_particles_inside_stair(particles, floor_id, rng)
            else:
                transition_state = "floor_entry"
                destination_floor = str(active_transition["to_floor"])
                transfer_particles_between_stairs(
                    particles, weights, floor_id, destination_floor, rng
                )
                floor_id = destination_floor
                map_data = load_map_constraints(floor_id)

                # Validate immediately on the destination map without cloning
                # every valid particle. Formal resampling is handled by N_eff.
                valid_destination = apply_map_constraint(particles, weights, map_data)
                if valid_destination == 0:
                    particles[:, :2] = sample_uniform_in_stair(
                        floor_id, len(particles), rng
                    )
                    weights.fill(1.0 / len(weights))
                else:
                    normalize(weights)

                active_transition = None
                transition_index += 1
        else:
            previous_particles = particles.copy()
            previous_weights = weights.copy()

            relative_heading_deg = float(getattr(step, "heading_relative_deg", step.heading_deg))
            heading_mode = str(getattr(step, "heading_mode", "stable"))
            heading_noise_deg, heading_drift_deg = predict(
                particles,
                float(step.step_length_m),
                relative_heading_deg,
                heading_mode,
                rng,
            )

            valid_particles_after_prediction = apply_map_constraint(
                particles, weights, map_data
            )
            rollback_occurred = False
            if valid_particles_after_prediction == 0:
                particles[:] = previous_particles
                weights[:] = previous_weights
                valid_particles_after_prediction = int(
                    valid_particle_mask(particles, map_data).sum()
                )
                rollback_occurred = True

            observations = nearby_ble(ble, step_time)
            observations = dict(
                sorted(
                    observations.items(),
                    key=lambda item: item[1],
                    reverse=True,
                )[:BLE_BEACONS_USED]
            )
            if int(step.step_id) % 2 == 0:
                update_with_ble(particles, weights, observations, beacons)
            normalize(weights)

        # BLE is deliberately skipped while the floor is ambiguous.
        if active_transition is not None or transition_state == "floor_entry":
            observations = {}
            normalize(weights)

        x_m, y_m = estimate_position(particles, weights, map_data)
        x_px, y_px = meter_to_pixel(x_m, y_m)

        output.append({
            "run_id": RUN_ID,
            "floor": floor_id,
            "transition_state": transition_state,
            "step_id": int(step.step_id),
            "time_sec": step_time,
            "x_m": x_m,
            "y_m": y_m,
            "x_px": x_px,
            "y_px": y_px,
            "heading_deg": weighted_map_heading_deg(
                particles, weights, float(getattr(step, "heading_relative_deg", step.heading_deg))
            ),
            "sensor_heading_deg": float(step.heading_deg),
            "relative_heading_deg": float(getattr(step, "heading_relative_deg", step.heading_deg)),
            "heading_mode": str(getattr(step, "heading_mode", "stable")),
            "turn_probability": float(getattr(step, "turn_probability", 0.0)),
            "turn_model": str(getattr(step, "turn_model", "none")),
            "heading_noise_deg": float(heading_noise_deg),
            "heading_alignment_drift_deg": float(heading_drift_deg),
            "step_length_m": float(step.step_length_m),
            "step_length_weinberg_m": float(getattr(step, "step_length_weinberg_m", step.step_length_m)),
            "step_length_regression_m": float(getattr(step, "step_length_regression_m", step.step_length_m)),
            "step_length_model": str(getattr(step, "step_length_model", "weinberg")),
            "valid_particles_after_prediction": int(valid_particles_after_prediction),
            "rollback_occurred": bool(rollback_occurred),
            "ble_beacons_used": len(observations),
            "source_stair_particles": (
                transition_source_particle_count
                if transition_state == "stairs"
                else 0
            ),
        })

        # Retain particle cloud output expected by teammate animation.
        particle_states.append(
            np.column_stack((particles[:, 0], particles[:, 1], weights))
        )

        effective_count = 1.0 / np.sum(weights ** 2)
        resampled = bool(
            active_transition is None
            and effective_count < NEFF_RESAMPLE_RATIO * N_PARTICLES
        )
        output[-1]["n_eff_before_resampling"] = float(effective_count)
        output[-1]["n_eff_ratio"] = float(effective_count / N_PARTICLES)
        output[-1]["resampled"] = resampled
        output[-1]["initial_heading_candidates_deg"] = ";".join(
            f"{angle:.0f}" for angle, _ in initial_heading_candidates
        )
        if resampled:
            systematic_resample(particles, weights, rng)

    return output, particle_states



def run_particle_filter():
    """Run the full particle filter pipeline for the currently active run and write positions, transitions, and errors to disk."""
    rng = np.random.default_rng(42)

    steps = load_steps()
    ble = load_ble()
    imu = load_imu()
    beacons = load_beacons()

    if USE_START_POSITION:
        start_position = load_start_position()
    else:
        map_data = load_map_constraints(FLOOR_ID)
        particles, _ = initialize_particles_from_map(map_data, rng)
        start_position = (float(particles[0, 0]), float(particles[0, 1]))

    transitions, transition_diagnostics = detect_floor_transitions(
        steps, imu, ble, FLOOR_ID
    )
    output, states = run_autonomous_particle_filter(
        steps,
        ble,
        beacons,
        FLOOR_ID,
        start_position,
        transitions,
        rng,
    )

    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    output_frame = pd.DataFrame(output)
    output_frame.to_csv(OUTPUT_FILE, index=False)

    # Save the full cloud in the same format expected by teammate animation.
    np.savez_compressed(
        ROOT / "data" / "processed" / RUN_ID / "particle_cloud.npz",
        states=np.asarray(states),
    )

    # Keep a master output plus one CSV per detected floor. This prevents any
    # floor-0/floor-1 path from being overlaid on a single map.
    for stale_file in OUTPUT_FILE.parent.glob(
        "particle_filter_positions_floor*.csv"
    ):
        stale_file.unlink()
    for assigned_floor, floor_rows in output_frame.groupby(
        output_frame["floor"].astype(str)
    ):
        floor_file = OUTPUT_FILE.with_name(
            f"particle_filter_positions_floor{assigned_floor}.csv"
        )
        floor_rows.to_csv(floor_file, index=False)

    transition_rows = pd.DataFrame(transitions)
    if transition_rows.empty:
        transition_rows = pd.DataFrame(columns=[
            "run_id",
            "start_time_sec",
            "end_time_sec",
            "from_floor",
            "to_floor",
            "confidence",
            "post_destination_margin_db",
            "ble_margin_change_db",
            "imu_votes",
        ])
    transition_rows.to_csv(TRANSITION_FILE, index=False)
    transition_diagnostics.to_csv(
        TRANSITION_FILE.with_name("floor_transition_diagnostics.csv"),
        index=False,
    )

    errors = calculate_errors(output)
    global GLOBAL_PF_ERRORS
    GLOBAL_PF_ERRORS[RUN_ID] = errors

    print(f"Run: {RUN_ID}")
    print(f"Steps processed: {len(output)}")
    print(f"Initial map-heading hypotheses: {output_frame['initial_heading_candidates_deg'].iloc[0]}")
    print(f"Systematic resampling events: {int(output_frame['resampled'].sum())}")
    print(f"Minimum N_eff before resampling: {output_frame['n_eff_before_resampling'].min():.1f}")
    print(f"Detected floor transitions: {len(transitions)}")
    for transition in transitions:
        print(
            f"  {transition['from_floor']} -> {transition['to_floor']}: "
            f"{transition['start_time_sec']:.2f}s to "
            f"{transition['end_time_sec']:.2f}s "
            f"(confidence={transition['confidence']:.2f})"
        )

    if errors.empty:
        return

    rmse_px = math.sqrt((errors['error_px'] ** 2).mean())
    rmse_m = math.sqrt((errors['error_m'] ** 2).mean())

    print(
        f"Mean error: {errors['error_px'].mean():.1f} px / "
        f"{errors['error_m'].mean():.2f} m"
    )
    print(
        f"RMSE:       {rmse_px:.1f} px / "
        f"{rmse_m:.2f} m"
    )
    print(
        f"Max error:  {errors['error_px'].max():.1f} px / "
        f"{errors['error_m'].max():.2f} m\n"
    )



if __name__ == "__main__":
    for run_id in RUN_IDS:
        set_run(run_id)
        run_particle_filter()

    summary = calculate_overall_errors(RUN_IDS)
    global GLOBAL_PF_ERROR_SUMMARY
    GLOBAL_PF_ERROR_SUMMARY = summary
    overall = summary[summary["run_id"] == "OVERALL"].iloc[0]

## 14. Localization Pipeline

The complete pipeline is:

**Raw CSV data** → **cleaning and synchronization** → **step detection and PDR** → **BLE processing and weighting** → **particle-filter prediction, BLE update, and map constraint** → **evaluation against reference points**

The notebook generates two trajectories from the same detected steps. The PDR-only trajectory uses IMU data and acts as the baseline, so its drift remains visible. The fused trajectory uses the same step lengths and headings inside the particle filter, then applies BLE likelihood updates and map constraints. Therefore, any difference between both results comes from the BLE correction and building constraints rather than from different step detections.

*(A visual flow diagram can be added here.)*


## 15. Experimental Design and Evaluation

**Evaluation procedure.** For each manually recorded checkpoint, we select the closest particle-filter estimate in time on the same floor. Before matching, the filter time is shifted using the first cleaned IMU timestamp. This accounts for the settling time between starting the recording and detecting the first step.

**Error metrics.** The pixel error for checkpoint $k$ is:

$$e_k^{\text{px}} = \sqrt{(x_k^{\text{px}} - x_{k,\text{ref}}^{\text{px}})^2 + (y_k^{\text{px}} - y_{k,\text{ref}}^{\text{px}})^2}$$

It is converted to metres using the floor-map scale. We then calculate:

$$\text{Mean error} = \frac1K\sum_{k=1}^{K} e_k, \qquad \text{RMSE} = \sqrt{\frac1K\sum_{k=1}^{K} e_k^2}, \qquad \text{Max error} = \max_k e_k$$

We focus mainly on RMSE because it gives a stronger penalty to large localization failures, such as accumulated drift or an incorrect floor transition.

**Ground-truth comparison.** The checkpoints were recorded independently by observing when the participant passed a known door. They were not used to correct the estimated route, apart from the known starting point. The later checkpoints are used only to measure the final localization error.


## 16. Results

For each run, the particle-filter code saves the estimated trajectory, floor-transition diagnostics, full particle cloud, and localization error at every reference timestamp. The cells below first show a static trajectory for each detected floor, followed by the particle-cloud animation.

The error table printed by the particle-filter cell contains the numerical results from the current execution. Reference markers are shown in the plots only for comparison and do not pull or correct the estimated path.


### 16.1 Floor-Aware Particle-Filter Trajectories

Each detected floor is plotted on its corresponding floor map. Colour shows the progress of time along the estimated route. Reference checkpoints and beacon positions are included only to help interpret the result. Since R3 contains a floor transition, its floor 0 and floor 1 trajectories are shown separately instead of being placed on the same map.


In [ ]:
# Static PF trajectories for all runs and all detected floors.
from pathlib import Path
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

map_files = {
    "0": Path("Floormap_0.png"),
    "1": Path("Floormap_1.png"),
}
references = pd.read_csv("data/metadata/reference_points.csv")
beacon_table = pd.read_csv("data/metadata/beacon_locations.csv")

for run_id in RUN_IDS:
    pf_path = Path("data/processed") / run_id / "particle_filter_positions.csv"
    pf = pd.read_csv(pf_path)
    for column in ["time_sec", "x_px", "y_px", "x_m", "y_m"]:
        pf[column] = pd.to_numeric(pf[column], errors="coerce")
    pf["floor"] = pf["floor"].astype(str)
    floors = list(dict.fromkeys(pf["floor"].tolist()))

    fig, axes = plt.subplots(1, len(floors), figsize=(8 * len(floors), 5.4), squeeze=False)
    fig.suptitle(f"{run_id}: particle-filter trajectory by floor", fontsize=14)

    for ax, floor in zip(axes[0], floors):
        floor_path = map_files.get(floor)
        if floor_path is None or not floor_path.exists():
            ax.text(0.5, 0.5, f"No floor map configured for floor {floor}", ha="center", va="center")
            ax.axis("off")
            continue

        image = mpimg.imread(floor_path)
        ax.imshow(image)
        floor_pf = pf[pf["floor"].eq(floor)].dropna(subset=["x_px", "y_px", "time_sec"])
        ax.plot(floor_pf["x_px"], floor_pf["y_px"], linewidth=1.5, alpha=0.65, label="PF path")
        time_scatter = ax.scatter(
            floor_pf["x_px"],
            floor_pf["y_px"],
            c=floor_pf["time_sec"],
            s=22,
            zorder=3,
        )
        if not floor_pf.empty:
            ax.scatter(floor_pf.iloc[0]["x_px"], floor_pf.iloc[0]["y_px"], s=85, label="floor segment start", zorder=4)
            ax.scatter(floor_pf.iloc[-1]["x_px"], floor_pf.iloc[-1]["y_px"], marker="X", s=95, label="floor segment end", zorder=4)

        run_refs = references[
            references["run_id"].astype(str).eq(run_id)
            & references["floor"].astype(str).eq(floor)
        ].copy()
        if not run_refs.empty:
            ax.scatter(run_refs["x"], run_refs["y"], marker="x", s=70, linewidths=2, label="reference checkpoints", zorder=5)
            for row in run_refs.itertuples():
                ax.annotate(str(row.ref_id), (row.x, row.y), xytext=(4, 4), textcoords="offset points", fontsize=8)

        floor_beacons = beacon_table[beacon_table["floor"].astype(str).eq(floor)]
        if not floor_beacons.empty:
            ax.scatter(floor_beacons["x"], floor_beacons["y"], marker="o", s=45, label="BLE beacons", zorder=5)
            for row in floor_beacons.itertuples():
                ax.annotate(str(row.beacon_id), (row.x, row.y), xytext=(4, -10), textcoords="offset points", fontsize=7)

        ax.set_title(f"Floor {floor}")
        ax.set_xlabel("x pixel")
        ax.set_ylabel("y pixel")
        ax.set_xlim(0, image.shape[1])
        ax.set_ylim(image.shape[0], 0)
        ax.legend(fontsize=8, loc="best")
        fig.colorbar(time_scatter, ax=ax, fraction=0.035, pad=0.02, label="time, s")

    fig.tight_layout()
    plt.show()

# Present the evaluation table directly in the notebook.
summary = GLOBAL_PF_ERROR_SUMMARY.copy()
display(summary)


In [ ]:
from __future__ import annotations
#!/usr/bin/env python3
"""Animate the current particle-filter trajectory output."""

import argparse
import csv
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib_cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(Path.cwd() / ".cache"))

import matplotlib

matplotlib.use("Agg")
import matplotlib.animation as animation
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np


# The animation reuses ROOT, coordinate constants, and meter_to_pixel() from
# the particle-filter section above. Keeping this notebook self-contained
# avoids requiring a separate scripts package in the submission folder.


VIDEO_DIR = ROOT / "data" / "outputs" / "videos"
GENERATE_PARTICLE_ANIMATIONS = False
DEFAULT_MAP_PATH = ROOT / "Floormap_0.png"

MAP_FILES = {
    "0": ROOT / "Floormap_0.png",
    "1": ROOT / "floormap_1.png",
    "S1": ROOT / "FL_S.png",
}


def read_rows(path: Path) -> list[dict[str, object]]:
    """Read a particle-filter positions CSV into a list of typed row dictionaries."""
    rows: list[dict[str, object]] = []
    with path.open(newline="") as handle:
        for row in csv.DictReader(handle):
            rows.append(
                {
                    "run_id": row.get("run_id", RUN_ID),
                    "floor": row.get("floor", ""),
                    "step_id": int(float(row["step_id"])),
                    "time_sec": float(row["time_sec"]),
                    "x_m": float(row["x_m"]),
                    "y_m": float(row["y_m"]),
                    "x_px": float(row["x_px"]),
                    "y_px": float(row["y_px"]),
                    "heading_deg": float(row["heading_deg"]),
                    "step_length_m": float(row["step_length_m"]),
                    "ble_beacons_used": int(float(row["ble_beacons_used"])),
                }
            )
    if not rows:
        raise ValueError(f"No particle-filter rows found in {path}")
    return rows


def padded_limits(values: list[float], lower: float | None = None, upper: float | None = None) -> tuple[float, float]:
    """Return (lower, upper) axis limits for a list of values, padded by a margin and including optional fixed bounds."""
    finite = [value for value in values if np.isfinite(value)]
    if lower is not None:
        finite.append(lower)
    if upper is not None:
        finite.append(upper)
    if not finite:
        return 0.0, 1.0

    lo = min(finite)
    hi = max(finite)
    if np.isclose(lo, hi):
        return lo - 1.0, hi + 1.0
    margin = 0.08 * (hi - lo)
    return lo - margin, hi + margin


def setup_meter_axes(ax, rows: list[dict[str, object]], run_id: str) -> None:
    """Configure a matplotlib axis for plotting the trajectory in metre coordinates."""
    xs = [float(row["x_m"]) for row in rows]
    ys = [float(row["y_m"]) for row in rows]
    ax.set_title(f"{run_id} Particle Filter Animation")
    ax.set_xlabel("x, m")
    ax.set_ylabel("y, m")
    ax.set_xlim(*padded_limits(xs, 0.0, ROOM_WIDTH_M))
    ax.set_ylim(*padded_limits(ys, 0.0, ROOM_HEIGHT_M))
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.25)
    ax.axvspan(0, ROOM_WIDTH_M, ymin=0, ymax=1, color="#4e79a7", alpha=0.04)


def setup_pixel_axes(ax, rows: list[dict[str, object]], run_id: str) -> None:
    """Configure a matplotlib axis for plotting the trajectory in raw pixel coordinates."""
    xs = [float(row["x_px"]) for row in rows]
    ys = [float(row["y_px"]) for row in rows]
    ax.set_title(f"{run_id} Particle Filter Pixel Animation")
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")
    ax.set_xlim(*padded_limits(xs, MAP_LEFT_PX, MAP_RIGHT_PX))
    y_lo, y_hi = padded_limits(ys, MAP_TOP_PX, MAP_BOTTOM_PX)
    ax.set_ylim(y_hi, y_lo)
    ax.grid(True, alpha=0.25)


def setup_map_axes(ax, rows: list[dict[str, object]], run_id: str, map_path: Path) -> None:
    """Configure a matplotlib axis for plotting the trajectory over the floor-plan image."""
    image = mpimg.imread(map_path)
    height, width = image.shape[:2]
    ax.imshow(image)
    ax.set_title(f"{run_id} Particle Filter On {map_path.name}")
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")
    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)
    ax.grid(True, alpha=0.18)

    xs = np.array([float(row["x_px"]) for row in rows])
    ys = np.array([float(row["y_px"]) for row in rows])
    outside = int(((xs < 0) | (xs > width) | (ys < 0) | (ys > height)).sum())
    if outside:
        ax.text(
            0.02,
            0.04,
            f"{outside} estimates outside map bounds",
            transform=ax.transAxes,
            fontsize=9,
            bbox={"facecolor": "white", "alpha": 0.78, "edgecolor": "#dddddd"},
        )


def save_animation(
    rows: list[dict[str, object]],
    output_path: Path,
    fps: int,
    view: str,
    map_path: Path,
    particle_states: np.ndarray | None = None,
) -> Path:
    """Build a matplotlib FuncAnimation of the particle-filter trajectory (and particle cloud) and return it as HTML."""
    run_id = str(rows[0]["run_id"] or RUN_ID)
    fig, ax = plt.subplots(figsize=(10, 7))

    if view == "map":
        if not map_path.exists():
            raise FileNotFoundError(f"Map image not found: {map_path}")
        setup_map_axes(ax, rows, run_id, map_path)
        x_field = "x_px"
        y_field = "y_px"
    elif view == "pixels":
        setup_pixel_axes(ax, rows, run_id)
        x_field = "x_px"
        y_field = "y_px"
    else:
        setup_meter_axes(ax, rows, run_id)
        x_field = "x_m"
        y_field = "y_m"

    xs = [float(row[x_field]) for row in rows]
    ys = [float(row[y_field]) for row in rows]

    # Initialize scatter for particle cloud
    cloud_scatter = None
    if particle_states is not None:
        cloud_scatter = ax.scatter([], [], s=10, c=[], cmap="viridis", alpha=0.5, zorder=3, edgecolors="none")

    path_line, = ax.plot([], [], color="#4e79a7", linewidth=2.2, label="PF path", zorder=4)
    current_dot, = ax.plot([], [], "o", color="#e15759", markersize=8, label="current estimate", zorder=5)
    start_dot = ax.scatter([xs[0]], [ys[0]], color="#59a14f", edgecolor="white", s=95, zorder=6, label="start")
    end_dot = ax.scatter([xs[-1]], [ys[-1]], color="#e15759", marker="X", edgecolor="white", s=110, zorder=6, label="end")
    info = ax.text(
        0.02,
        0.96,
        "",
        transform=ax.transAxes,
        va="top",
        fontsize=9,
        bbox={"facecolor": "white", "alpha": 0.78, "edgecolor": "#dddddd"},
    )
    ax.legend(loc="best", fontsize=8)

    def update(frame: int):
        """Advance the animation to the given frame, updating the path, current position marker, particle cloud, and info text."""
        row = rows[frame]
        
        if cloud_scatter is not None:
            # state is (N_PARTICLES, 3) -> x_m, y_m, weight
            state = particle_states[frame]
            p_xs = state[:, 0]
            p_ys = state[:, 1]
            p_weights = state[:, 2]
            
            if view == "map" or view == "pixels":
                p_xs, p_ys = meter_to_pixel(p_xs, p_ys)
                
            cloud_scatter.set_offsets(np.c_[p_xs, p_ys])
            cloud_scatter.set_array(p_weights)
        
        path_line.set_data(xs[: frame + 1], ys[: frame + 1])
        current_dot.set_data([xs[frame]], [ys[frame]])
        info.set_text(
            f"t = {float(row['time_sec']):.2f} s\n"
            f"step = {int(row['step_id'])}\n"
            f"heading = {float(row['heading_deg']):.1f} deg\n"
            f"step length = {float(row['step_length_m']):.2f} m\n"
            f"BLE beacons = {int(row['ble_beacons_used'])}"
        )
        artists = [path_line, current_dot, info, start_dot, end_dot]
        if cloud_scatter is not None:
            artists.append(cloud_scatter)
        return artists

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(rows),
        interval=1000 / fps,
        blit=True,
    )

    jshtml = ani.to_jshtml()

    plt.close(fig)

    return jshtml


def main() -> None:
    """Discover particle-filter output CSVs, animate each run/floor, and display the resulting animations."""
    parser = argparse.ArgumentParser(
        description="Animate the CSV produced by the current scripts/run_particle_filter.py."
    )
    parser.add_argument(
        "--input",
        type=Path,
        help="Particle-filter CSV. Default: auto-discover all runs.",
    )
    parser.add_argument(
        "--output",
        type=Path,
        help="Output MP4 path. Default: data/outputs/videos/<run>_particle_filter_animation.mp4.",
    )
    parser.add_argument(
        "--view",
        choices=("meters", "pixels", "map"),
        default="map",
        help="Animation coordinate view. Default: map.",
    )
    parser.add_argument(
        "--map",
        type=Path,
        default=DEFAULT_MAP_PATH,
        help="Map image used when --view map is selected. Default: Floormap_0.png.",
    )
    parser.add_argument("--fps", type=int, default=8, help="Frames per second. Default: 8.")
    args = parser.parse_args(args=[])

    if args.input:
        input_paths = [args.input]
    else:
        PROCESSED_DIR = ROOT / "data" / "processed"
        input_paths = sorted(PROCESSED_DIR.glob("*/particle_filter_positions.csv"))

    if not input_paths:
        print("No particle_filter_positions.csv files found to animate.")
        return

    for input_path in input_paths:
        try:
            rows = read_rows(input_path)
        except Exception as e:
            print(f"Error reading {input_path}: {e}")
            continue

        if not rows:
            continue

        run_id = str(rows[0].get("run_id") or input_path.parent.name)
        npz_path = input_path.parent / "particle_cloud.npz"
        all_particle_states = None
        if npz_path.exists():
            try:
                all_particle_states = np.load(npz_path)["states"]
                if len(all_particle_states) != len(rows):
                    print(f"Warning: Particle cloud steps ({len(all_particle_states)}) does not match rows ({len(rows)}). Ignoring cloud.")
                    all_particle_states = None
            except Exception as e:
                print(f"Failed to load particle cloud: {e}")

        from collections import defaultdict
        rows_by_floor = defaultdict(list)
        indices_by_floor = defaultdict(list)
        for idx, row in enumerate(rows):
            floor = str(row.get("floor", ""))
            rows_by_floor[floor].append(row)
            indices_by_floor[floor].append(idx)

        for floor, floor_rows in rows_by_floor.items():
            floor_suffix = f"_floor{floor}" if floor else ""
            map_path = MAP_FILES.get(floor, args.map)
            
            floor_particle_states = None
            if all_particle_states is not None:
                floor_particle_states = all_particle_states[indices_by_floor[floor]]

            if args.output and len(rows_by_floor) == 1 and len(input_paths) == 1:
                output_path = args.output
            elif args.output and len(input_paths) == 1:
                output_path = args.output.with_name(f"{args.output.stem}{floor_suffix}{args.output.suffix}")
            else:
                output_path = VIDEO_DIR / f"{run_id}{floor_suffix}_particle_filter_animation.mp4"

            jshtml = save_animation(floor_rows, output_path, args.fps, args.view, map_path, floor_particle_states)
            print(f"Animated {len(floor_rows)} estimates for floor '{floor}' from {input_path}")
            from IPython.display import display, HTML
            display(HTML(jshtml))


if __name__ == "__main__":
    if GENERATE_PARTICLE_ANIMATIONS:
        main()
    else:
        print(
            "Particle-cloud animation generation is skipped during Run All to keep the notebook fast. "
            "Set GENERATE_PARTICLE_ANIMATIONS = True in this cell to create and display the MP4 animations."
        )


## 17. Related Work and Academic Positioning

We selected three sources that directly match the main parts of this project: PDR, map-constrained multi-floor particle filtering, and BLE-based position information.

The author names and paper titles are linked to their publication pages. DOI links are also included in Section 21.

| Source | Main contribution | How it relates to our work | Main difference |
|---|---|---|---|
| [Harle (2013)](https://www.cl.cam.ac.uk/~rkh23/site/publication/pub35/) [1] | Reviews pedestrian inertial positioning, including step-and-heading PDR, map information, particle filters, and absolute corrections. It explains why PDR normally needs external correction over longer distances. | Sections 8 to 10 build our smartphone PDR. Sections 11 to 13 add BLE and map constraints to reduce drift. | This source is a broad survey, while our work implements and evaluates one small smartphone-based pipeline. |
| [Woodman and Harle (2008)](https://www.cl.cam.ac.uk/~rkh23/site/publication/pub0/) [2] | Combines inertial pedestrian tracking, a detailed building model, particle filtering, multiple floors, stairways, and radio-based initialization. | Our walkable polygons remove particles that cross walls, and our staircase detector transfers the particle cloud between floors. | Their system used a foot-mounted IMU and a larger multi-floor test area. We use a handheld smartphone, simpler polygons, BLE updates, and a known starting point. |
| [Faragher and Harle (2015)](https://www.cl.cam.ac.uk/~rkh23/site/publication/pub42/) [3] | Studies BLE indoor positioning and the effects of fast fading and beacon density. Their main method uses radio fingerprinting. | Our RSSI plots show the same instability. Therefore, we treat BLE as probabilistic evidence rather than as an exact distance measurement. | We use a log-distance likelihood with at most two beacons instead of building a dense fingerprint map. |

### Academic Positioning

This project is a small-scale engineering implementation and evaluation. We do not claim to introduce a new localization algorithm. The main contribution is a reproducible pipeline that starts from raw smartphone recordings and combines:

- IMU and BLE cleaning and synchronization,
- step-and-heading PDR,
- BLE RSSI likelihood weighting,
- building constraints inside a particle filter,
- automatic staircase-transition handling, and
- checkpoint-based evaluation across four runs.

The results are mainly useful for comparing the behaviour of R1 to R4 within our own setup. A direct numerical comparison with the three papers would not be fair because their sensors, beacon density, map size, initialization, and ground-truth methods are different. Instead, our notebook shows how established methods can be combined into one complete and transparent indoor-localization system.


## 18. Discussion

Use the outputs of the current execution as the result for the current inputs. The original submission README and saved executed notebook described different datasets. The repository's [experiment record](docs/EXPERIMENTS.md) identifies the verified source and inputs. Start checkpoints are included in the implementation's aggregate, while unavailable same-floor checkpoints are skipped; neither this aggregate nor nearest-time floor-conditioned matching establishes autonomous floor accuracy.


## 19. Error Sources, Limitations, and Improvements

Manual checkpoint timing, sparse BLE coverage, phone pose, map scaling and stochastic particle collapse limit accuracy. Centered signal windows, full-run pseudo-label fitting, future BLE windows and the floor-transition detector prevent a strict real-time claim. Calibration labels were derived by matching old-session movement bouts to map route segments; their provenance needs independent review. Existing field logs, map scale fields and stale metadata disagree in places. Future work should validate these inputs, use held-out buildings and participants, and evaluate missed floors as failures rather than skipping them.


## 20. Conclusion

This Group E coursework prototype reconstructs pedestrian trajectories from smartphone IMU and BLE recordings using PDR, building geometry and particle filtering. Ridge step-length regression and diagnostic logistic turn probabilities extend the physical motion model. Reproducibility depends on authorized local data and the documented environment. The accompanying documentation records the evidence and limits of the current snapshot.


## 21. References

1. Harle, R. (2013). [“A Survey of Indoor Inertial Positioning Systems for Pedestrians.”](https://www.cl.cam.ac.uk/~rkh23/site/publication/pub35/) *IEEE Communications Surveys & Tutorials*, 15(3), 1281–1293. [DOI: 10.1109/SURV.2012.121912.00075](https://doi.org/10.1109/SURV.2012.121912.00075).
2. Woodman, O. J., & Harle, R. K. (2008). [“Pedestrian Localisation for Indoor Environments.”](https://www.cl.cam.ac.uk/~rkh23/site/publication/pub0/) *Proceedings of the 10th International Conference on Ubiquitous Computing (UbiComp 2008)*, 114–123. [DOI: 10.1145/1409635.1409651](https://doi.org/10.1145/1409635.1409651).
3. Faragher, R., & Harle, R. (2015). [“Location Fingerprinting With Bluetooth Low Energy Beacons.”](https://www.cl.cam.ac.uk/~rkh23/site/publication/pub42/) *IEEE Journal on Selected Areas in Communications*, 33(11), 2418–2428. [DOI: 10.1109/JSAC.2015.2430281](https://doi.org/10.1109/JSAC.2015.2430281).

**Software and project resources.** The project includes NumPy, pandas, Matplotlib, the complete Group E recordings, floor maps, beacon metadata, and manually recorded reference timestamps. These resources support reproducibility and are not counted as related-work sources.
